# Cassava Leaf Disease Classification — fill-in baseline

Problem family: **Multiclass image classification**. Official metric: **Accuracy**.

**Learning objective:** Small scratch CNN + cross entropy.

[Dataset and original
task](https://www.kaggle.com/competitions/cassava-leaf-disease-classification) · [Primary
solution
evidence](https://www.kaggle.com/competitions/cassava-leaf-disease-classification/discussion/221957)

The linked first-place write-up reports an ensemble including EfficientNet-B4. This baseline is
a small scratch CNN; the advanced companion uses one EfficientNet-B4. Neither reproduces the
full ensemble, imports CropNet, or adds external disease data.

**Watch for:** The official score is accuracy. Report macro F1 diagnostically, but do not
replace the checkpoint-selection metric.

**Standalone:** this file contains every project-specific helper it uses. Only installed
third-party libraries and your local data/assets are required; no repository checkout, source
imports, project installation or external bootstrap is used. Only the optional dataset cell
needs internet when enabled; all training works offline. Edit DATA_DIR and
output/storage settings in stage 1. Train media on local disk, then persist your output bundle.

This notebook is a learning exercise, not a claimed leaderboard reproduction. No trained
weights or real Kaggle data are bundled. Keep grading answers closed. Use only explicitly
permitted resources. Internet/prepared-code access is not assumed on contest day.

Complete the named exercises and their nearby checks in order. Setup and safety helpers are
supplied.

After the lesson, close both notebooks and rebuild the essential route from memory. See the
repository's blank-notebook playbook for the final transfer test.

## How to use this lesson

Complete one named exercise, run its nearby checks, then interpret the result. Supplied cells
contain setup or integration; edit only configuration or marked exercise bodies. Every baseline
code cell retains a collapsed full reference immediately afterward. Use Hint 1, then Hint 2,
before opening a reference.

By the end you should be able to state the data/metric contract, choose a legal split, construct
one correct model input, validate a saved candidate and deliver an ID-aligned file. Mini-checks
use tiny separate examples or the current training batch; they do not certify real-data quality.

 | Stage | Goal |
|---|---|
| 1 | Environment and configuration |
| 1a | Optional dataset acquisition |
| 2 | Understand and load the data contract |
| 3 | Inspect examples and distributions |
| 4 | Build and test the validation split |
| 5 | Measure imbalance and choose a response |
| 6 | Build features or one correct batch |
| 7 | Connect outputs, loss and official metric |
| 8 | Train and preserve a recoverable candidate |
| 9 | Interpret validation and compare experiments |
| 10 | Reload and infer every test item |
| 11 | Deliver and verify the submission |



## 1. Environment and configuration

Run the supplied imports and runtime helpers in order. Edit paths, time budget and model policy
in the configuration cells. Inspect the actual device; CPU checks cannot establish GPU
throughput. Supplied storage/safety definitions are folded where the viewer supports it: run
them, but reconstruct the short algorithmic functions during closed-book practice.



In [ ]:
import json
import pickle
import random
import shutil
import sys
import time
import zipfile
from collections.abc import Callable, Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
from typing import Any
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader

# ruff: noqa: E402



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import json
import pickle
import random
import shutil
import sys
import time
import zipfile
from collections.abc import Callable, Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
from typing import Any
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader

# ruff: noqa: E402

</code></pre>
</details>


### Supplied helper — `CompetitionConfig`

Explicit notebook state; no installed project is required.




In [ ]:
@dataclass
class CompetitionConfig:
    """Explicit notebook state; no installed project is required."""

    data_dir: Path
    output_dir: Path
    seed: int = 42
    num_workers: int = 0
    pretrained_allowed: bool = False
    time_budget_minutes: float = 60
    device: str = "auto"

    def prepare(self) -> None:
        self.device = (
            ("cuda" if torch.cuda.is_available() else "cpu")
            if self.device == "auto"
            else self.device
        )
        self.output_dir.mkdir(parents=True, exist_ok=True)
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def to_dict(self) -> dict:
        return {**asdict(self), "data_dir": str(self.data_dir), "output_dir": str(self.output_dir)}




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass
class CompetitionConfig:
    &quot;&quot;&quot;Explicit notebook state; no installed project is required.&quot;&quot;&quot;

    data_dir: Path
    output_dir: Path
    seed: int = 42
    num_workers: int = 0
    pretrained_allowed: bool = False
    time_budget_minutes: float = 60
    device: str = &quot;auto&quot;

    def prepare(self) -&gt; None:
        self.device = (
            (&quot;cuda&quot; if torch.cuda.is_available() else &quot;cpu&quot;)
            if self.device == &quot;auto&quot;
            else self.device
        )
        self.output_dir.mkdir(parents=True, exist_ok=True)
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def to_dict(self) -&gt; dict:
        return {**asdict(self), &quot;data_dir&quot;: str(self.data_dir), &quot;output_dir&quot;: str(self.output_dir)}


</code></pre>
</details>


### Supplied helper — `TimeBudget`

Leave delivery time available; a running batch/fit cannot be interrupted safely.




In [ ]:
class TimeBudget:
    """Leave delivery time available; a running batch/fit cannot be interrupted safely."""

    def __init__(self, minutes: float, reserve_minutes: float = 10) -> None:
        if not 0 <= reserve_minutes < minutes:
            raise ValueError("Require 0 <= reserve < total minutes")
        self.deadline = time.monotonic() + 60 * (minutes - reserve_minutes)

    @property
    def should_stop(self) -> bool:
        return time.monotonic() >= self.deadline

    def checkpoint(self, stage: str) -> None:
        if self.should_stop:
            raise TimeoutError(f"Delivery reserve reached before {stage}")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>class TimeBudget:
    &quot;&quot;&quot;Leave delivery time available; a running batch/fit cannot be interrupted safely.&quot;&quot;&quot;

    def __init__(self, minutes: float, reserve_minutes: float = 10) -&gt; None:
        if not 0 &lt;= reserve_minutes &lt; minutes:
            raise ValueError(&quot;Require 0 &lt;= reserve &lt; total minutes&quot;)
        self.deadline = time.monotonic() + 60 * (minutes - reserve_minutes)

    @property
    def should_stop(self) -&gt; bool:
        return time.monotonic() &gt;= self.deadline

    def checkpoint(self, stage: str) -&gt; None:
        if self.should_stop:
            raise TimeoutError(f&quot;Delivery reserve reached before {stage}&quot;)


</code></pre>
</details>


### Supplied helper — `runtime_paths`

Media stays on local disk; optional durable storage is an explicit destination.




In [ ]:
def runtime_paths(
    *, runtime_dir: Path | None = None, persistent_dir: Path | None = None
) -> SimpleNamespace:
    """Media stays on local disk; optional durable storage is an explicit destination."""
    root = (
        Path(runtime_dir)
        if runtime_dir is not None
        else (Path("/content/olp_runtime") if "google.colab" in sys.modules else Path.cwd())
    )
    data, output = root / "data", root / "outputs"
    data.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(
        data_dir=data,
        output_dir=output,
        persistent_dir=Path(persistent_dir) if persistent_dir is not None else None,
    )




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def runtime_paths(
    *, runtime_dir: Path | None = None, persistent_dir: Path | None = None
) -&gt; SimpleNamespace:
    &quot;&quot;&quot;Media stays on local disk; optional durable storage is an explicit destination.&quot;&quot;&quot;
    root = (
        Path(runtime_dir)
        if runtime_dir is not None
        else (Path(&quot;/content/olp_runtime&quot;) if &quot;google.colab&quot; in sys.modules else Path.cwd())
    )
    data, output = root / &quot;data&quot;, root / &quot;outputs&quot;
    data.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(
        data_dir=data,
        output_dir=output,
        persistent_dir=Path(persistent_dir) if persistent_dir is not None else None,
    )


</code></pre>
</details>


### Supplied helper — `mount_google_drive`

Call only when an explicitly configured Colab path uses Drive.




In [ ]:
def mount_google_drive() -> None:
    """Call only when an explicitly configured Colab path uses Drive."""
    from google.colab import drive

    drive.mount("/content/drive")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def mount_google_drive() -&gt; None:
    &quot;&quot;&quot;Call only when an explicitly configured Colab path uses Drive.&quot;&quot;&quot;
    from google.colab import drive

    drive.mount(&quot;/content/drive&quot;)


</code></pre>
</details>


### Supplied helper — `stage_data`

Copy a folder or extract a ZIP once, then train from the local destination.




In [ ]:
def stage_data(source: Path | str, destination: Path | str) -> Path:
    """Copy a folder or extract a ZIP once, then train from the local destination."""
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or destination.is_relative_to(source):
        raise ValueError("Staging destination must be separate from its source")
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError("Reuse staged DATA_DIR and set DATA_ARCHIVE=None")
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError("Archive contains a path outside the destination")
            archive.extractall(destination)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def stage_data(source: Path | str, destination: Path | str) -&gt; Path:
    &quot;&quot;&quot;Copy a folder or extract a ZIP once, then train from the local destination.&quot;&quot;&quot;
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or destination.is_relative_to(source):
        raise ValueError(&quot;Staging destination must be separate from its source&quot;)
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(&quot;Reuse staged DATA_DIR and set DATA_ARCHIVE=None&quot;)
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError(&quot;Archive contains a path outside the destination&quot;)
            archive.extractall(destination)
    return destination


</code></pre>
</details>


### Supplied helper — `gpu_report`

Inspect the actual runtime before choosing image size, batch size or precision.




In [ ]:
def gpu_report() -> dict:
    """Inspect the actual runtime before choosing image size, batch size or precision."""
    if not torch.cuda.is_available():
        return {"device": "cpu", "precision": "float32"}
    free, total = torch.cuda.mem_get_info()
    return {
        "device": torch.cuda.get_device_name(),
        "free_GiB": free / 2**30,
        "total_GiB": total / 2**30,
        "precision": "bf16" if torch.cuda.is_bf16_supported() else "fp16",
    }




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def gpu_report() -&gt; dict:
    &quot;&quot;&quot;Inspect the actual runtime before choosing image size, batch size or precision.&quot;&quot;&quot;
    if not torch.cuda.is_available():
        return {&quot;device&quot;: &quot;cpu&quot;, &quot;precision&quot;: &quot;float32&quot;}
    free, total = torch.cuda.mem_get_info()
    return {
        &quot;device&quot;: torch.cuda.get_device_name(),
        &quot;free_GiB&quot;: free / 2**30,
        &quot;total_GiB&quot;: total / 2**30,
        &quot;precision&quot;: &quot;bf16&quot; if torch.cuda.is_bf16_supported() else &quot;fp16&quot;,
    }


</code></pre>
</details>


### Configure this run

Edit these values before proceeding. DATA_DIR points at your staged pack; choose a separate
empty folder for an online download. Keep model permission explicit, start with a small row/time
budget, and let the actual device report guide neural batch size. Each run writes to a new
candidate directory.



In [ ]:
FAMILY, VARIANT = "04_leaves", "baseline"
DATA_ARCHIVE, PERSISTENT_DIR = None, None
if "google.colab" in sys.modules and any(
    "/content/drive/" in str(value) for value in (DATA_ARCHIVE, PERSISTENT_DIR)
):
    mount_google_drive()
paths = runtime_paths(persistent_dir=PERSISTENT_DIR)
DATA_DIR = (
    paths.data_dir / "learning" / FAMILY
    if "google.colab" in sys.modules
    else Path.cwd() / "data" / "learning" / FAMILY
)
if DATA_ARCHIVE is not None:
    stage_data(DATA_ARCHIVE, DATA_DIR)
RUN_ID = f"{FAMILY}-{VARIANT}-{uuid4().hex[:8]}"
OUTPUT = paths.output_dir / "learning" / RUN_ID
DURABLE_OUTPUT = paths.persistent_dir / RUN_ID if paths.persistent_dir is not None else None
PRETRAINED_ALLOWED, WEIGHT_PATH = False, None
SEED, MAX_ROWS, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 6000, 128, 16, 3
TIME_BUDGET_MINUTES, DELIVERY_RESERVE_MINUTES = 60, 10
GROUP_COLUMN, TIME_COLUMN = None, None
USE_IMBALANCE_WEIGHTS = False
MODEL_NAME = "scratch_cnn"
IMAGE_SUFFIX = ".png"



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>FAMILY, VARIANT = &quot;04_leaves&quot;, &quot;baseline&quot;
DATA_ARCHIVE, PERSISTENT_DIR = None, None
if &quot;google.colab&quot; in sys.modules and any(
    &quot;/content/drive/&quot; in str(value) for value in (DATA_ARCHIVE, PERSISTENT_DIR)
):
    mount_google_drive()
paths = runtime_paths(persistent_dir=PERSISTENT_DIR)
DATA_DIR = (
    paths.data_dir / &quot;learning&quot; / FAMILY
    if &quot;google.colab&quot; in sys.modules
    else Path.cwd() / &quot;data&quot; / &quot;learning&quot; / FAMILY
)
if DATA_ARCHIVE is not None:
    stage_data(DATA_ARCHIVE, DATA_DIR)
RUN_ID = f&quot;{FAMILY}-{VARIANT}-{uuid4().hex[:8]}&quot;
OUTPUT = paths.output_dir / &quot;learning&quot; / RUN_ID
DURABLE_OUTPUT = paths.persistent_dir / RUN_ID if paths.persistent_dir is not None else None
PRETRAINED_ALLOWED, WEIGHT_PATH = False, None
SEED, MAX_ROWS, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 6000, 128, 16, 3
TIME_BUDGET_MINUTES, DELIVERY_RESERVE_MINUTES = 60, 10
GROUP_COLUMN, TIME_COLUMN = None, None
USE_IMBALANCE_WEIGHTS = False
MODEL_NAME = &quot;scratch_cnn&quot;
IMAGE_SUFFIX = &quot;.png&quot;

</code></pre>
</details>


### Configure this run

Edit these values before proceeding. DATA_DIR points at your staged pack; choose a separate
empty folder for an online download. Keep model permission explicit, start with a small row/time
budget, and let the actual device report guide neural batch size. Each run writes to a new
candidate directory.



In [ ]:
ASSET_PROVENANCE = (
    ""  # exact asset/version/license; required when enabling local pretrained features
)
competition = CompetitionConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT,
    seed=SEED,
    num_workers=0,
    pretrained_allowed=PRETRAINED_ALLOWED,
    time_budget_minutes=TIME_BUDGET_MINUTES,
)
competition.prepare()
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)
print(gpu_report())
print("Local data:", DATA_DIR, "New candidate:", OUTPUT)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>ASSET_PROVENANCE = (
    &quot;&quot;  # exact asset/version/license; required when enabling local pretrained features
)
competition = CompetitionConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT,
    seed=SEED,
    num_workers=0,
    pretrained_allowed=PRETRAINED_ALLOWED,
    time_budget_minutes=TIME_BUDGET_MINUTES,
)
competition.prepare()
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)
print(gpu_report())
print(&quot;Local data:&quot;, DATA_DIR, &quot;New candidate:&quot;, OUTPUT)

</code></pre>
</details>


## 1a. Optional dataset acquisition

The cell is skipped by default. For online preparation, first accept this competition's
rules on its Kaggle page and configure credentials using the
[official authentication
guide](https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication).
Set DOWNLOAD_DATA=True below. Local uv sync includes the official Kaggle client. In a copied
notebook/Colab runtime without it, install only `kaggle>=2.2,<3` before running this cell;
do not install this project or replace Colab's torch/NumPy/CUDA stack.

Use KAGGLE_API_TOKEN in your environment, the client's configured token/legacy credential
file, or enable PROMPT_FOR_KAGGLE_TOKEN for hidden input. Never paste a token into saved code.
No rules are accepted automatically, no submission is uploaded, and no weights are fetched.
Downloads can be several GB: allow space for the cached ZIP plus extracted data. DATA_DIR
must be a dedicated empty folder (or a completed download from this cell). Existing manual
data/fixtures are left untouched; keep DOWNLOAD_DATA=False to use them.

This cell reuses a completed dataset/cache, expands nested train/test/CSV archives, and
checks the expected layout. On Kaggle itself, attach competition data in the notebook UI,
point DATA_DIR at its mounted folder, and leave this switch off. For restricted-room drills,
prepare data beforehand and keep this cell disabled. The training timer restarts afterward.



### Supplied helper — `_json_default`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, "tolist"):
        return value.tolist()
    raise TypeError(f"Cannot serialize {type(value).__name__} as JSON")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _json_default(value: Any) -&gt; Any:
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, &quot;tolist&quot;):
        return value.tolist()
    raise TypeError(f&quot;Cannot serialize {type(value).__name__} as JSON&quot;)


</code></pre>
</details>


### Supplied helper — `_write`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _write(value: Any, path: Path | str, kind: str) -> Path:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.{uuid4().hex}.tmp")
    try:
        if kind == "json":
            temporary.write_text(
                json.dumps(
                    value, default=_json_default, indent=2, ensure_ascii=False, allow_nan=False
                ),
                encoding="utf-8",
            )
        elif kind == "pickle":
            with temporary.open("wb") as handle:
                pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            import torch

            torch.save(value, temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _write(value: Any, path: Path | str, kind: str) -&gt; Path:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f&quot;.{destination.name}.{uuid4().hex}.tmp&quot;)
    try:
        if kind == &quot;json&quot;:
            temporary.write_text(
                json.dumps(
                    value, default=_json_default, indent=2, ensure_ascii=False, allow_nan=False
                ),
                encoding=&quot;utf-8&quot;,
            )
        elif kind == &quot;pickle&quot;:
            with temporary.open(&quot;wb&quot;) as handle:
                pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            import torch

            torch.save(value, temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


</code></pre>
</details>


### Supplied helper — `save_json`

Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values.




In [ ]:
def save_json(value: Any, path: Path | str) -> Path:
    """Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values."""
    return _write(value, path, "json")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def save_json(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values.&quot;&quot;&quot;
    return _write(value, path, &quot;json&quot;)


</code></pre>
</details>


In [ ]:
import os
import subprocess
from getpass import getpass
from importlib.util import find_spec

DOWNLOAD_DATA = False
KAGGLE_COMPETITION = "cassava-leaf-disease-classification"
EXPECTED_DATA_PATHS = ("train.csv", "sample_submission.csv", "train_images", "test_images")
DOWNLOAD_CACHE = DATA_DIR.parent / "_downloads" / KAGGLE_COMPETITION
DOWNLOAD_TIMEOUT_SECONDS = 7200
PROMPT_FOR_KAGGLE_TOKEN = "google.colab" in sys.modules



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import os
import subprocess
from getpass import getpass
from importlib.util import find_spec

DOWNLOAD_DATA = False
KAGGLE_COMPETITION = &quot;cassava-leaf-disease-classification&quot;
EXPECTED_DATA_PATHS = (&quot;train.csv&quot;, &quot;sample_submission.csv&quot;, &quot;train_images&quot;, &quot;test_images&quot;)
DOWNLOAD_CACHE = DATA_DIR.parent / &quot;_downloads&quot; / KAGGLE_COMPETITION
DOWNLOAD_TIMEOUT_SECONDS = 7200
PROMPT_FOR_KAGGLE_TOKEN = &quot;google.colab&quot; in sys.modules

</code></pre>
</details>


### Supplied helper — `extract_dataset_zip`

Extract data without allowing absolute paths, traversal or silent overwrites.




In [ ]:
def extract_dataset_zip(source: Path, destination: Path) -> None:
    """Extract data without allowing absolute paths, traversal or silent overwrites."""
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > shutil.disk_usage(destination).free:
            raise OSError("Insufficient disk space for the next archive")
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Archive contains a path outside the dataset folder")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Duplicated paths indicate a wrong archive/layout; preserve the first file.
            with archive.open(member) as incoming, target.open("xb") as outgoing:
                shutil.copyfileobj(incoming, outgoing)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def extract_dataset_zip(source: Path, destination: Path) -&gt; None:
    &quot;&quot;&quot;Extract data without allowing absolute paths, traversal or silent overwrites.&quot;&quot;&quot;
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) &gt; shutil.disk_usage(destination).free:
            raise OSError(&quot;Insufficient disk space for the next archive&quot;)
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(&quot;Archive contains a path outside the dataset folder&quot;)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Duplicated paths indicate a wrong archive/layout; preserve the first file.
            with archive.open(member) as incoming, target.open(&quot;xb&quot;) as outgoing:
                shutil.copyfileobj(incoming, outgoing)


</code></pre>
</details>


### Supplied helper — `download_practice_data`

Use the official CLI, then publish the complete extracted folder by renaming it.




In [ ]:
def download_practice_data(
    competition: str,
    destination: Path,
    cache: Path,
    expected_paths: Sequence[str],
    *,
    prompt_token: bool = False,
    timeout_seconds: int = 7200,
) -> Path:
    """Use the official CLI, then publish the complete extracted folder by renaming it."""
    if not competition or any(
        char not in "abcdefghijklmnopqrstuvwxyz0123456789-" for char in competition
    ):
        raise ValueError("Expected a Kaggle competition slug, not a URL or shell command")
    destination, cache = Path(destination).resolve(), Path(cache).resolve()
    if (
        destination == cache
        or cache.is_relative_to(destination)
        or destination.is_relative_to(cache)
    ):
        raise ValueError("Dataset and download cache must be separate folders")
    marker = destination / ".kaggle-source.json"
    if marker.is_file():
        source = json.loads(marker.read_text(encoding="utf-8"))
        if source.get("competition") == competition and all(
            (destination / path).exists() for path in expected_paths
        ):
            print("Reusing completed dataset:", destination)
            return destination
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            "DATA_DIR is not empty/completed; choose a new folder or keep DOWNLOAD_DATA=False"
        )
    cache.mkdir(parents=True, exist_ok=True)
    bundle = cache / f"{competition}.zip"
    if not bundle.is_file() or not zipfile.is_zipfile(bundle):
        if find_spec("kaggle") is None:
            raise ModuleNotFoundError(
                "Install kaggle>=2.2,<3 for online preparation; "
                "local users can run uv sync --locked"
            )
        if prompt_token and not os.environ.get("KAGGLE_API_TOKEN"):
            token = getpass("Kaggle API token (blank uses configured credentials): ").strip()
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
        # No shell, embedded credentials, auto-acceptance, uploads or package installation.
        command = [
            sys.executable,
            "-m",
            "kaggle",
            "competitions",
            "download",
            competition,
            "--path",
            str(cache),
            "--quiet",
        ]
        if bundle.exists():
            command.append("--force")  # A partial/invalid archive must be fetched again.
        result = subprocess.run(
            command,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            timeout=timeout_seconds,
            check=False,
        )
        if result.returncode:
            # Do not put client output, possible signed URLs or credentials in saved notebooks.
            raise RuntimeError(
                f"Kaggle download failed (exit {result.returncode}). "
                "Check credentials, competition access/rules, internet and disk space. "
                "No dataset was published."
            )
        if not bundle.is_file() or not zipfile.is_zipfile(bundle):
            raise ValueError("Kaggle did not produce a complete competition ZIP; check the cache")
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(f".{destination.name}-incoming-{uuid4().hex[:8]}")
    staging.mkdir()
    try:
        extract_dataset_zip(bundle, staging)
        # TGS has train.zip with images/ and masks/; some tasks also wrap CSVs in ZIPs.
        for nested in sorted(staging.rglob("*.zip")):
            folder = nested.parent
            if nested.stem in {"train", "test", "train_images", "test_images"}:
                with zipfile.ZipFile(nested) as archive:
                    names = [item.filename for item in archive.infolist() if not item.is_dir()]
                if names and not all(name.startswith(nested.stem + "/") for name in names):
                    folder = nested.parent / nested.stem
            extract_dataset_zip(nested, folder)
            nested.unlink()  # The outer cached bundle remains available for recovery.
        missing = [path for path in expected_paths if not (staging / path).exists()]
        if missing:
            raise FileNotFoundError(f"Unexpected dataset layout; missing: {missing}")
        save_json(
            {
                "competition": competition,
                "source": f"https://www.kaggle.com/competitions/{competition}/data",
                "expected_paths": list(expected_paths),
            },
            staging / marker.name,
        )
        if destination.exists():
            destination.rmdir()  # Only the previously checked empty directory.
        staging.replace(destination)
    except Exception as error:
        raise RuntimeError(
            "Dataset preparation failed; DATA_DIR was not replaced. "
            f"Inspect partial extraction at {staging}"
        ) from error
    print("Prepared dataset:", destination)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def download_practice_data(
    competition: str,
    destination: Path,
    cache: Path,
    expected_paths: Sequence[str],
    *,
    prompt_token: bool = False,
    timeout_seconds: int = 7200,
) -&gt; Path:
    &quot;&quot;&quot;Use the official CLI, then publish the complete extracted folder by renaming it.&quot;&quot;&quot;
    if not competition or any(
        char not in &quot;abcdefghijklmnopqrstuvwxyz0123456789-&quot; for char in competition
    ):
        raise ValueError(&quot;Expected a Kaggle competition slug, not a URL or shell command&quot;)
    destination, cache = Path(destination).resolve(), Path(cache).resolve()
    if (
        destination == cache
        or cache.is_relative_to(destination)
        or destination.is_relative_to(cache)
    ):
        raise ValueError(&quot;Dataset and download cache must be separate folders&quot;)
    marker = destination / &quot;.kaggle-source.json&quot;
    if marker.is_file():
        source = json.loads(marker.read_text(encoding=&quot;utf-8&quot;))
        if source.get(&quot;competition&quot;) == competition and all(
            (destination / path).exists() for path in expected_paths
        ):
            print(&quot;Reusing completed dataset:&quot;, destination)
            return destination
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            &quot;DATA_DIR is not empty/completed; choose a new folder or keep DOWNLOAD_DATA=False&quot;
        )
    cache.mkdir(parents=True, exist_ok=True)
    bundle = cache / f&quot;{competition}.zip&quot;
    if not bundle.is_file() or not zipfile.is_zipfile(bundle):
        if find_spec(&quot;kaggle&quot;) is None:
            raise ModuleNotFoundError(
                &quot;Install kaggle&gt;=2.2,&lt;3 for online preparation; &quot;
                &quot;local users can run uv sync --locked&quot;
            )
        if prompt_token and not os.environ.get(&quot;KAGGLE_API_TOKEN&quot;):
            token = getpass(&quot;Kaggle API token (blank uses configured credentials): &quot;).strip()
            if token:
                os.environ[&quot;KAGGLE_API_TOKEN&quot;] = token
        # No shell, embedded credentials, auto-acceptance, uploads or package installation.
        command = [
            sys.executable,
            &quot;-m&quot;,
            &quot;kaggle&quot;,
            &quot;competitions&quot;,
            &quot;download&quot;,
            competition,
            &quot;--path&quot;,
            str(cache),
            &quot;--quiet&quot;,
        ]
        if bundle.exists():
            command.append(&quot;--force&quot;)  # A partial/invalid archive must be fetched again.
        result = subprocess.run(
            command,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            timeout=timeout_seconds,
            check=False,
        )
        if result.returncode:
            # Do not put client output, possible signed URLs or credentials in saved notebooks.
            raise RuntimeError(
                f&quot;Kaggle download failed (exit {result.returncode}). &quot;
                &quot;Check credentials, competition access/rules, internet and disk space. &quot;
                &quot;No dataset was published.&quot;
            )
        if not bundle.is_file() or not zipfile.is_zipfile(bundle):
            raise ValueError(&quot;Kaggle did not produce a complete competition ZIP; check the cache&quot;)
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(f&quot;.{destination.name}-incoming-{uuid4().hex[:8]}&quot;)
    staging.mkdir()
    try:
        extract_dataset_zip(bundle, staging)
        # TGS has train.zip with images/ and masks/; some tasks also wrap CSVs in ZIPs.
        for nested in sorted(staging.rglob(&quot;*.zip&quot;)):
            folder = nested.parent
            if nested.stem in {&quot;train&quot;, &quot;test&quot;, &quot;train_images&quot;, &quot;test_images&quot;}:
                with zipfile.ZipFile(nested) as archive:
                    names = [item.filename for item in archive.infolist() if not item.is_dir()]
                if names and not all(name.startswith(nested.stem + &quot;/&quot;) for name in names):
                    folder = nested.parent / nested.stem
            extract_dataset_zip(nested, folder)
            nested.unlink()  # The outer cached bundle remains available for recovery.
        missing = [path for path in expected_paths if not (staging / path).exists()]
        if missing:
            raise FileNotFoundError(f&quot;Unexpected dataset layout; missing: {missing}&quot;)
        save_json(
            {
                &quot;competition&quot;: competition,
                &quot;source&quot;: f&quot;https://www.kaggle.com/competitions/{competition}/data&quot;,
                &quot;expected_paths&quot;: list(expected_paths),
            },
            staging / marker.name,
        )
        if destination.exists():
            destination.rmdir()  # Only the previously checked empty directory.
        staging.replace(destination)
    except Exception as error:
        raise RuntimeError(
            &quot;Dataset preparation failed; DATA_DIR was not replaced. &quot;
            f&quot;Inspect partial extraction at {staging}&quot;
        ) from error
    print(&quot;Prepared dataset:&quot;, destination)
    return destination


</code></pre>
</details>


In [ ]:
if DOWNLOAD_DATA:
    if DATA_ARCHIVE is not None:
        raise ValueError("Choose either DATA_ARCHIVE staging or DOWNLOAD_DATA, then rerun setup")
    DATA_DIR = download_practice_data(
        KAGGLE_COMPETITION,
        DATA_DIR,
        DOWNLOAD_CACHE,
        EXPECTED_DATA_PATHS,
        prompt_token=PROMPT_FOR_KAGGLE_TOKEN,
        timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    )
source_marker = DATA_DIR / ".kaggle-source.json"
if source_marker.is_file():
    save_json(json.loads(source_marker.read_text(encoding="utf-8")), OUTPUT / "data_source.json")
print("Download enabled:", DOWNLOAD_DATA, "Dataset folder:", DATA_DIR)
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>if DOWNLOAD_DATA:
    if DATA_ARCHIVE is not None:
        raise ValueError(&quot;Choose either DATA_ARCHIVE staging or DOWNLOAD_DATA, then rerun setup&quot;)
    DATA_DIR = download_practice_data(
        KAGGLE_COMPETITION,
        DATA_DIR,
        DOWNLOAD_CACHE,
        EXPECTED_DATA_PATHS,
        prompt_token=PROMPT_FOR_KAGGLE_TOKEN,
        timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    )
source_marker = DATA_DIR / &quot;.kaggle-source.json&quot;
if source_marker.is_file():
    save_json(json.loads(source_marker.read_text(encoding=&quot;utf-8&quot;)), OUTPUT / &quot;data_source.json&quot;)
print(&quot;Download enabled:&quot;, DOWNLOAD_DATA, &quot;Dataset folder:&quot;, DATA_DIR)
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)

</code></pre>
</details>


## 2. Understand and load the data contract

One cassava image has one of five class labels. image_id links the image to a submission row.
Preserve the recorded label-to-index mapping; the model emits raw logits, while the submission
requires original labels.

Read the first rows, column names and actual shapes below. Expected: unique nonempty prediction
IDs; matching test/sample ID sets; a target schema that agrees with the official output. Dataset
size depends on your local files and MAX_ROWS.



In [ ]:
ID_COLUMN, TARGET_COLUMNS = "image_id", ["label"]
train = pd.read_csv(DATA_DIR / "train.csv", converters={ID_COLUMN: str})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", converters={ID_COLUMN: str})
test = sample[[ID_COLUMN]].copy()
train["image"] = "train_images/" + train[ID_COLUMN]
test["image"] = "test_images/" + test[ID_COLUMN]
if MAX_ROWS is not None and len(train) > MAX_ROWS:
    train = train.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
for name, frame in (("train", train), ("test", test), ("sample", sample)):
    if (
        frame[ID_COLUMN].isna().any()
        or frame[ID_COLUMN].eq("").any()
        or frame[ID_COLUMN].duplicated().any()
    ):
        raise ValueError(f"{name} must have unique, nonempty prediction-unit IDs")
assert not set(train[ID_COLUMN]) & set(test[ID_COLUMN])
assert set(test[ID_COLUMN]) == set(sample[ID_COLUMN])
assert sample.columns.tolist() == [ID_COLUMN, *TARGET_COLUMNS]
print(train.head(3).to_string(index=False))
print("Shapes:", train.shape, test.shape, sample.shape)
assert all((DATA_DIR / path).is_file() for path in train["image"].head(10))
assert all((DATA_DIR / path).is_file() for path in test["image"])



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>ID_COLUMN, TARGET_COLUMNS = &quot;image_id&quot;, [&quot;label&quot;]
train = pd.read_csv(DATA_DIR / &quot;train.csv&quot;, converters={ID_COLUMN: str})
sample = pd.read_csv(DATA_DIR / &quot;sample_submission.csv&quot;, converters={ID_COLUMN: str})
test = sample[[ID_COLUMN]].copy()
train[&quot;image&quot;] = &quot;train_images/&quot; + train[ID_COLUMN]
test[&quot;image&quot;] = &quot;test_images/&quot; + test[ID_COLUMN]
if MAX_ROWS is not None and len(train) &gt; MAX_ROWS:
    train = train.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
for name, frame in ((&quot;train&quot;, train), (&quot;test&quot;, test), (&quot;sample&quot;, sample)):
    if (
        frame[ID_COLUMN].isna().any()
        or frame[ID_COLUMN].eq(&quot;&quot;).any()
        or frame[ID_COLUMN].duplicated().any()
    ):
        raise ValueError(f&quot;{name} must have unique, nonempty prediction-unit IDs&quot;)
assert not set(train[ID_COLUMN]) &amp; set(test[ID_COLUMN])
assert set(test[ID_COLUMN]) == set(sample[ID_COLUMN])
assert sample.columns.tolist() == [ID_COLUMN, *TARGET_COLUMNS]
print(train.head(3).to_string(index=False))
print(&quot;Shapes:&quot;, train.shape, test.shape, sample.shape)
assert all((DATA_DIR / path).is_file() for path in train[&quot;image&quot;].head(10))
assert all((DATA_DIR / path).is_file() for path in test[&quot;image&quot;])

</code></pre>
</details>


In [ ]:
assert len(train) > 0 and len(test) == len(sample)
assert train[ID_COLUMN].is_unique and sample[ID_COLUMN].is_unique
print("Data-contract checks passed: prediction units and sample coverage.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert len(train) &gt; 0 and len(test) == len(sample)
assert train[ID_COLUMN].is_unique and sample[ID_COLUMN].is_unique
print(&quot;Data-contract checks passed: prediction units and sample coverage.&quot;)

</code></pre>
</details>


### Stop and explain

What is the prediction unit, and which columns may enter the model?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

One cassava image has one of five class labels. image_id links the image to a submission row.
Preserve the recorded label-to-index mapping; the model emits raw logits, while the submission
requires original labels.

</details>



## 3. Inspect examples and distributions

Inspect the plotted examples before selecting a model. Write observations before reading the
next section.

Can you recognize class differences in the image grid? Are image dimensions and color channels
consistent? Could several leaves from one plant, farm or acquisition session appear in both
folds?



In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flat, train.head(8).iterrows(), strict=False):
    with Image.open(DATA_DIR / row["image"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"id={row[ID_COLUMN]}; label={row['label']}")
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUTPUT / "inspection.png")
plt.show()
plt.close(fig)
print("Sizes:", [Image.open(DATA_DIR / path).size for path in train["image"].head(8)])
print("Class counts:", train["label"].value_counts().to_dict())



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flat, train.head(8).iterrows(), strict=False):
    with Image.open(DATA_DIR / row[&quot;image&quot;]) as image:
        ax.imshow(image.convert(&quot;RGB&quot;))
    ax.set_title(f&quot;id={row[ID_COLUMN]}; label={row[&#x27;label&#x27;]}&quot;)
    ax.axis(&quot;off&quot;)
plt.tight_layout()
fig.savefig(OUTPUT / &quot;inspection.png&quot;)
plt.show()
plt.close(fig)
print(&quot;Sizes:&quot;, [Image.open(DATA_DIR / path).size for path in train[&quot;image&quot;].head(8)])
print(&quot;Class counts:&quot;, train[&quot;label&quot;].value_counts().to_dict())

</code></pre>
</details>


### Stop and explain

Record three observations: one distribution issue, one suspicious example, and one possible
leakage source. What experiment does each justify?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Use the actual plots and rows; there is no fixed answer independent of your pack. Convert each
observation into a falsifiable check before spending training time.

</details>



## 4. Build and test the validation split

Stratify independent images and keep acquisition identities together whenever supplied. Ensure
class support and freeze the class order. Image filenames alone do not prove that specimens are
independent.



In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split

</code></pre>
</details>


### Supplied helper — `SplitResult`

Training/validation tables plus their original positional indexes.




In [ ]:
@dataclass(slots=True)
class SplitResult:
    """Training/validation tables plus their original positional indexes."""

    train: pd.DataFrame
    valid: pd.DataFrame
    train_indices: np.ndarray
    valid_indices: np.ndarray




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass(slots=True)
class SplitResult:
    &quot;&quot;&quot;Training/validation tables plus their original positional indexes.&quot;&quot;&quot;

    train: pd.DataFrame
    valid: pd.DataFrame
    train_indices: np.ndarray
    valid_indices: np.ndarray


</code></pre>
</details>


### Supplied helper — `_stratification_key`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _stratification_key(frame: pd.DataFrame, targets: Sequence[str]) -> pd.Series:
    if not targets:
        raise ValueError("At least one target is required for stratification")
    if len(targets) == 1:
        return frame[targets[0]].astype(str)
    return frame[list(targets)].astype(str).agg("|".join, axis=1)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _stratification_key(frame: pd.DataFrame, targets: Sequence[str]) -&gt; pd.Series:
    if not targets:
        raise ValueError(&quot;At least one target is required for stratification&quot;)
    if len(targets) == 1:
        return frame[targets[0]].astype(str)
    return frame[list(targets)].astype(str).agg(&quot;|&quot;.join, axis=1)


</code></pre>
</details>


### Supplied helper — `make_split`

Create a deterministic holdout with leakage-aware group and time options.




In [ ]:
def make_split(
    frame: pd.DataFrame,
    *,
    target_columns: str | Sequence[str] | None = None,
    group_column: str | None = None,
    time_column: str | None = None,
    valid_size: float = 0.2,
    seed: int = 42,
) -> SplitResult:
    """Create a deterministic holdout with leakage-aware group and time options."""
    if not 0 < valid_size < 1:
        raise ValueError("valid_size must be between 0 and 1")
    if len(frame) < 2:
        raise ValueError("At least two rows are required")
    targets = [target_columns] if isinstance(target_columns, str) else list(target_columns or [])
    for column in [*targets, group_column, time_column]:
        if column is not None and column not in frame.columns:
            raise KeyError(f"Missing split column: {column}")

    indices = np.arange(len(frame))
    if time_column:
        timestamps = pd.to_datetime(frame[time_column], errors="raise")
        if timestamps.isna().any():
            raise ValueError("Time split requires complete timestamps")
        order = np.argsort(timestamps.to_numpy(), kind="stable")
        ordered_times = timestamps.to_numpy()[order]
        boundaries = np.flatnonzero(ordered_times[1:] != ordered_times[:-1]) + 1
        if not len(boundaries):
            raise ValueError("Need at least two distinct times for a strict chronological split")
        desired = len(frame) * (1 - valid_size)
        cut = int(boundaries[np.argmin(abs(boundaries - desired))])
        train_idx, valid_idx = order[:cut], order[cut:]
    elif group_column and targets:
        folds = max(2, round(1 / valid_size))
        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        try:
            train_idx, valid_idx = next(
                splitter.split(indices, _stratification_key(frame, targets), frame[group_column])
            )
        except ValueError:
            fallback = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
            train_idx, valid_idx = next(fallback.split(indices, groups=frame[group_column]))
    elif group_column:
        splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
        train_idx, valid_idx = next(splitter.split(indices, groups=frame[group_column]))
    else:
        stratify = _stratification_key(frame, targets) if targets else None
        try:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=stratify
            )
        except ValueError:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=None
            )

    if group_column:
        overlap = set(frame.iloc[train_idx][group_column]) & set(
            frame.iloc[valid_idx][group_column]
        )
        if overlap:
            raise RuntimeError(f"Group leakage detected for {len(overlap)} groups")
    return SplitResult(
        train=frame.iloc[train_idx].reset_index(drop=True),
        valid=frame.iloc[valid_idx].reset_index(drop=True),
        train_indices=np.asarray(train_idx),
        valid_indices=np.asarray(valid_idx),
    )




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def make_split(
    frame: pd.DataFrame,
    *,
    target_columns: str | Sequence[str] | None = None,
    group_column: str | None = None,
    time_column: str | None = None,
    valid_size: float = 0.2,
    seed: int = 42,
) -&gt; SplitResult:
    &quot;&quot;&quot;Create a deterministic holdout with leakage-aware group and time options.&quot;&quot;&quot;
    if not 0 &lt; valid_size &lt; 1:
        raise ValueError(&quot;valid_size must be between 0 and 1&quot;)
    if len(frame) &lt; 2:
        raise ValueError(&quot;At least two rows are required&quot;)
    targets = [target_columns] if isinstance(target_columns, str) else list(target_columns or [])
    for column in [*targets, group_column, time_column]:
        if column is not None and column not in frame.columns:
            raise KeyError(f&quot;Missing split column: {column}&quot;)

    indices = np.arange(len(frame))
    if time_column:
        timestamps = pd.to_datetime(frame[time_column], errors=&quot;raise&quot;)
        if timestamps.isna().any():
            raise ValueError(&quot;Time split requires complete timestamps&quot;)
        order = np.argsort(timestamps.to_numpy(), kind=&quot;stable&quot;)
        ordered_times = timestamps.to_numpy()[order]
        boundaries = np.flatnonzero(ordered_times[1:] != ordered_times[:-1]) + 1
        if not len(boundaries):
            raise ValueError(&quot;Need at least two distinct times for a strict chronological split&quot;)
        desired = len(frame) * (1 - valid_size)
        cut = int(boundaries[np.argmin(abs(boundaries - desired))])
        train_idx, valid_idx = order[:cut], order[cut:]
    elif group_column and targets:
        folds = max(2, round(1 / valid_size))
        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        try:
            train_idx, valid_idx = next(
                splitter.split(indices, _stratification_key(frame, targets), frame[group_column])
            )
        except ValueError:
            fallback = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
            train_idx, valid_idx = next(fallback.split(indices, groups=frame[group_column]))
    elif group_column:
        splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
        train_idx, valid_idx = next(splitter.split(indices, groups=frame[group_column]))
    else:
        stratify = _stratification_key(frame, targets) if targets else None
        try:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=stratify
            )
        except ValueError:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=None
            )

    if group_column:
        overlap = set(frame.iloc[train_idx][group_column]) &amp; set(
            frame.iloc[valid_idx][group_column]
        )
        if overlap:
            raise RuntimeError(f&quot;Group leakage detected for {len(overlap)} groups&quot;)
    return SplitResult(
        train=frame.iloc[train_idx].reset_index(drop=True),
        valid=frame.iloc[valid_idx].reset_index(drop=True),
        train_indices=np.asarray(train_idx),
        valid_indices=np.asarray(valid_idx),
    )


</code></pre>
</details>


### Exercise 1 — `create_holdout`

Input: a row-per-prediction table and its actual leakage columns. Return: the split record and
independent training/validation copies. The union must retain all rows; identities must not
cross folds.

<details>
<summary>Hint 1 — choose the operation</summary>

Stratify independent images and keep acquisition identities together whenever supplied. Ensure
class support and freeze the class order. Image filenames alone do not prove that specimens are
independent.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Use make_split with valid_size=0.2; return copies of its train and valid tables.

</details>




In [ ]:
def create_holdout(
    frame: pd.DataFrame,
    target_columns: Any = None,
    group_column: str | None = None,
    time_column: str | None = None,
    seed: int = 42,
) -> Any:
    """Input: a row-per-prediction table and its actual leakage columns. Return: the split
    record and independent training/validation copies. The union must retain all rows;
    identities must not cross folds.
    """
    raise NotImplementedError('TODO: implement create_holdout using the stated contract')
    return split, tr, va




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def create_holdout(
    frame: pd.DataFrame,
    target_columns: Any = None,
    group_column: str | None = None,
    time_column: str | None = None,
    seed: int = 42,
) -&gt; Any:
    &quot;&quot;&quot;Input: a row-per-prediction table and its actual leakage columns. Return: the split
    record and independent training/validation copies. The union must retain all rows;
    identities must not cross folds.
    &quot;&quot;&quot;
    split = make_split(
        frame,
        target_columns=target_columns,
        group_column=group_column,
        time_column=time_column,
        valid_size=0.2,
        seed=seed,
    )
    tr, va = split.train.copy(), split.valid.copy()
    return split, tr, va


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_frame = pd.DataFrame({"id": range(12), "group": np.repeat(range(6), 2)})
toy_split, toy_train, toy_valid = create_holdout(toy_frame, group_column="group", seed=7)
assert len(toy_train) + len(toy_valid) == 12, "The split must retain every row"
assert not set(toy_train["group"]) & set(toy_valid["group"]), "Repeated identities leaked"
assert not set(toy_split.train_indices) & set(toy_split.valid_indices)
toy_again = create_holdout(toy_frame, group_column="group", seed=7)[0]
np.testing.assert_array_equal(toy_split.valid_indices, toy_again.valid_indices)
print("Split checks passed: complete, group-disjoint and reproducible.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_frame = pd.DataFrame({&quot;id&quot;: range(12), &quot;group&quot;: np.repeat(range(6), 2)})
toy_split, toy_train, toy_valid = create_holdout(toy_frame, group_column=&quot;group&quot;, seed=7)
assert len(toy_train) + len(toy_valid) == 12, &quot;The split must retain every row&quot;
assert not set(toy_train[&quot;group&quot;]) &amp; set(toy_valid[&quot;group&quot;]), &quot;Repeated identities leaked&quot;
assert not set(toy_split.train_indices) &amp; set(toy_split.valid_indices)
toy_again = create_holdout(toy_frame, group_column=&quot;group&quot;, seed=7)[0]
np.testing.assert_array_equal(toy_split.valid_indices, toy_again.valid_indices)
print(&quot;Split checks passed: complete, group-disjoint and reproducible.&quot;)

</code></pre>
</details>


In [ ]:
split, tr, va = create_holdout(
    train, target_columns="label", group_column=GROUP_COLUMN, time_column=TIME_COLUMN, seed=SEED
)
assert not set(tr[ID_COLUMN]) & set(va[ID_COLUMN])
if GROUP_COLUMN:
    assert not set(tr[GROUP_COLUMN]) & set(va[GROUP_COLUMN])
if TIME_COLUMN:
    assert pd.to_datetime(tr[TIME_COLUMN]).max() < pd.to_datetime(va[TIME_COLUMN]).min()
labels = sorted(train["label"].unique().tolist())
mapping = {label: index for index, label in enumerate(labels)}
assert set(tr["label"]) == set(labels)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>split, tr, va = create_holdout(
    train, target_columns=&quot;label&quot;, group_column=GROUP_COLUMN, time_column=TIME_COLUMN, seed=SEED
)
assert not set(tr[ID_COLUMN]) &amp; set(va[ID_COLUMN])
if GROUP_COLUMN:
    assert not set(tr[GROUP_COLUMN]) &amp; set(va[GROUP_COLUMN])
if TIME_COLUMN:
    assert pd.to_datetime(tr[TIME_COLUMN]).max() &lt; pd.to_datetime(va[TIME_COLUMN]).min()
labels = sorted(train[&quot;label&quot;].unique().tolist())
mapping = {label: index for index, label in enumerate(labels)}
assert set(tr[&quot;label&quot;]) == set(labels)

</code></pre>
</details>


In [ ]:
assert len(tr) > 0 and len(va) > 0
assert not set(tr[ID_COLUMN]) & set(va[ID_COLUMN])
print("Actual-fold checks passed: nonempty disjoint prediction IDs.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert len(tr) &gt; 0 and len(va) &gt; 0
assert not set(tr[ID_COLUMN]) &amp; set(va[ID_COLUMN])
print(&quot;Actual-fold checks passed: nonempty disjoint prediction IDs.&quot;)

</code></pre>
</details>


### Stop and explain

Would a random row split measure the generalization you need here? Identify the boundary that
would leak information.

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Stratify independent images and keep acquisition identities together whenever supplied. Ensure
class support and freeze the class order. Image filenames alone do not prove that specimens are
independent.

</details>



## 5. Measure imbalance and choose a response

Align inverse-frequency weights to the model's class-index order. Cross-entropy can use these
weights during training, but the official accuracy still reflects the original validation
distribution. Keep macro F1 as a diagnostic.

$w_c=N/(C n_c)$. Cross-entropy consumes [B,C] raw logits and [B] integer class indexes; accuracy
compares decoded labels.



In [ ]:
counts = tr["label"].value_counts().reindex(labels)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>counts = tr[&quot;label&quot;].value_counts().reindex(labels)

</code></pre>
</details>


### Exercise 2 — `class_weights`

Input: training counts already reindexed to saved class order. Return: float32 inverse-frequency
weights [C]. The smallest count should get the largest weight.

<details>
<summary>Hint 1 — choose the operation</summary>

Align inverse-frequency weights to the model's class-index order. Cross-entropy can use these
weights during training, but the official accuracy still reflects the original validation
distribution. Keep macro F1 as a diagnostic.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Compute N/(C*count_c) elementwise; do not sort weights by frequency.

</details>




In [ ]:
def class_weights(counts: pd.Series, training_rows: int, num_classes: int) -> Any:
    """Input: training counts already reindexed to saved class order. Return: float32 inverse-
    frequency weights [C]. The smallest count should get the largest weight.
    """
    raise NotImplementedError('TODO: implement class_weights using the stated contract')
    return weights




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def class_weights(counts: pd.Series, training_rows: int, num_classes: int) -&gt; Any:
    &quot;&quot;&quot;Input: training counts already reindexed to saved class order. Return: float32 inverse-
    frequency weights [C]. The smallest count should get the largest weight.
    &quot;&quot;&quot;
    weights = torch.tensor((training_rows / (num_classes * counts)).to_numpy(), dtype=torch.float32)
    return weights


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_weights = class_weights(pd.Series([6, 2], index=[0, 1]), 8, 2)
np.testing.assert_allclose(toy_weights.numpy(), [2 / 3, 2])
assert toy_weights.dtype == torch.float32
print("Class-weight checks passed: class-index order and inverse frequencies.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_weights = class_weights(pd.Series([6, 2], index=[0, 1]), 8, 2)
np.testing.assert_allclose(toy_weights.numpy(), [2 / 3, 2])
assert toy_weights.dtype == torch.float32
print(&quot;Class-weight checks passed: class-index order and inverse frequencies.&quot;)

</code></pre>
</details>


In [ ]:
weights = class_weights(counts, len(tr), len(labels))
print("Majority accuracy reference:", counts.max() / counts.sum())
print("Training counts:", counts.to_dict(), "Weights enabled:", USE_IMBALANCE_WEIGHTS)
counts.plot.bar(title="Training class support")
plt.tight_layout()
plt.savefig(OUTPUT / "imbalance.png")
plt.show()
plt.close()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>weights = class_weights(counts, len(tr), len(labels))
print(&quot;Majority accuracy reference:&quot;, counts.max() / counts.sum())
print(&quot;Training counts:&quot;, counts.to_dict(), &quot;Weights enabled:&quot;, USE_IMBALANCE_WEIGHTS)
counts.plot.bar(title=&quot;Training class support&quot;)
plt.tight_layout()
plt.savefig(OUTPUT / &quot;imbalance.png&quot;)
plt.show()
plt.close()

</code></pre>
</details>


### Stop and explain

Does changing training weights also change the official score or validation distribution? What
would justify enabling the response?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Align inverse-frequency weights to the model's class-index order. Cross-entropy can use these
weights during training, but the official accuracy still reflects the original validation
distribution. Keep macro F1 as a diagnostic.

</details>



## 6. Build features or one correct batch

The scratch baseline uses deterministic resizing for its first sanity run. Training and
evaluation transforms are separate objects even when initially identical. Inspect a [B,3,H,W]
batch and its integer class indexes.



### Worked component — `predict_logits`

Ordered, deterministic inference; BF16/FP16 on capable CUDA, FP32 on CPU.




In [ ]:
@torch.inference_mode()
def predict_logits(model: nn.Module, loader: Iterable, *, device: str = "cpu") -> np.ndarray:
    """Ordered, deterministic inference; BF16/FP16 on capable CUDA, FP32 on CPU."""
    model.to(device).eval()
    enabled = torch.device(device).type == "cuda"
    dtype = torch.bfloat16 if enabled and torch.cuda.is_bf16_supported() else torch.float16
    outputs = []
    for batch in loader:
        inputs = batch[0] if isinstance(batch, (tuple, list)) else batch
        with torch.autocast(torch.device(device).type, dtype=dtype, enabled=enabled):
            logits = model(inputs.to(device))
        outputs.append(logits.float().cpu().numpy())
    if not outputs:
        raise ValueError("Inference loader is empty")
    return np.concatenate(outputs)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@torch.inference_mode()
def predict_logits(model: nn.Module, loader: Iterable, *, device: str = &quot;cpu&quot;) -&gt; np.ndarray:
    &quot;&quot;&quot;Ordered, deterministic inference; BF16/FP16 on capable CUDA, FP32 on CPU.&quot;&quot;&quot;
    model.to(device).eval()
    enabled = torch.device(device).type == &quot;cuda&quot;
    dtype = torch.bfloat16 if enabled and torch.cuda.is_bf16_supported() else torch.float16
    outputs = []
    for batch in loader:
        inputs = batch[0] if isinstance(batch, (tuple, list)) else batch
        with torch.autocast(torch.device(device).type, dtype=dtype, enabled=enabled):
            logits = model(inputs.to(device))
        outputs.append(logits.float().cpu().numpy())
    if not outputs:
        raise ValueError(&quot;Inference loader is empty&quot;)
    return np.concatenate(outputs)


</code></pre>
</details>


In [ ]:
from torch.utils.data import Dataset
from torchvision.transforms import v2



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from torch.utils.data import Dataset
from torchvision.transforms import v2

</code></pre>
</details>


### Worked component — `ImageTableDataset`

Load RGB images and optional class indexes from a pandas annotation table.




In [ ]:
class ImageTableDataset(Dataset[Any]):
    """Load RGB images and optional class indexes from a pandas annotation table."""

    def __init__(
        self,
        frame: pd.DataFrame,
        *,
        image_column: str,
        target_column: str | None = None,
        root: Path | str = ".",
        transform: Callable[[Image.Image], torch.Tensor] | None = None,
        label_to_index: Mapping[object, int] | None = None,
    ) -> None:
        self.frame = frame.reset_index(drop=True)
        self.image_column = image_column
        self.target_column = target_column
        self.root = Path(root)
        self.transform = transform or build_image_transforms(training=False)
        self.label_to_index = dict(label_to_index or {})

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> Any:
        row = self.frame.iloc[index]
        path = self.root / str(row[self.image_column])
        with Image.open(path) as source:
            image = source.convert("RGB")
        tensor = self.transform(image)
        if self.target_column:
            label = row[self.target_column]
            target = torch.tensor(self.label_to_index.get(label, label), dtype=torch.long)
            return tensor, target
        return tensor




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>class ImageTableDataset(Dataset[Any]):
    &quot;&quot;&quot;Load RGB images and optional class indexes from a pandas annotation table.&quot;&quot;&quot;

    def __init__(
        self,
        frame: pd.DataFrame,
        *,
        image_column: str,
        target_column: str | None = None,
        root: Path | str = &quot;.&quot;,
        transform: Callable[[Image.Image], torch.Tensor] | None = None,
        label_to_index: Mapping[object, int] | None = None,
    ) -&gt; None:
        self.frame = frame.reset_index(drop=True)
        self.image_column = image_column
        self.target_column = target_column
        self.root = Path(root)
        self.transform = transform or build_image_transforms(training=False)
        self.label_to_index = dict(label_to_index or {})

    def __len__(self) -&gt; int:
        return len(self.frame)

    def __getitem__(self, index: int) -&gt; Any:
        row = self.frame.iloc[index]
        path = self.root / str(row[self.image_column])
        with Image.open(path) as source:
            image = source.convert(&quot;RGB&quot;)
        tensor = self.transform(image)
        if self.target_column:
            label = row[self.target_column]
            target = torch.tensor(self.label_to_index.get(label, label), dtype=torch.long)
            return tensor, target
        return tensor


</code></pre>
</details>


### Worked component — `build_image_transforms`

Build conservative training or deterministic validation transforms.




In [ ]:
def build_image_transforms(
    *,
    size: int = 224,
    training: bool,
    normalize: bool = True,
) -> Callable[[Image.Image], torch.Tensor]:
    """Build conservative training or deterministic validation transforms."""
    operations: list[Any]
    if training:
        operations = [
            v2.RandomResizedCrop((size, size), scale=(0.7, 1.0)),
            v2.RandomHorizontalFlip(),
            v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        ]
    else:
        operations = [v2.Resize((size, size))]
    operations.append(v2.ToImage())
    operations.append(v2.ToDtype(torch.float32, scale=True))
    if normalize:
        operations.append(v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)))
    return v2.Compose(operations)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def build_image_transforms(
    *,
    size: int = 224,
    training: bool,
    normalize: bool = True,
) -&gt; Callable[[Image.Image], torch.Tensor]:
    &quot;&quot;&quot;Build conservative training or deterministic validation transforms.&quot;&quot;&quot;
    operations: list[Any]
    if training:
        operations = [
            v2.RandomResizedCrop((size, size), scale=(0.7, 1.0)),
            v2.RandomHorizontalFlip(),
            v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        ]
    else:
        operations = [v2.Resize((size, size))]
    operations.append(v2.ToImage())
    operations.append(v2.ToDtype(torch.float32, scale=True))
    if normalize:
        operations.append(v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)))
    return v2.Compose(operations)


</code></pre>
</details>


### Exercise 3 — `make_transform_pair`

Input: square image size. Return separate train/evaluation transform objects. This first scratch
baseline is deterministic; an augmentation experiment changes only the training transform.

<details>
<summary>Hint 1 — choose the operation</summary>

The scratch baseline uses deterministic resizing for its first sanity run. Training and
evaluation transforms are separate objects even when initially identical. Inspect a [B,3,H,W]
batch and its integer class indexes.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Use build_image_transforms with training=False for the initial sanity run; evaluation must
remain deterministic.

</details>




In [ ]:
def make_transform_pair(size: int) -> Any:
    """Input: square image size. Return separate train/evaluation transform objects. This first
    scratch baseline is deterministic; an augmentation experiment changes only the training
    transform.
    """
    raise NotImplementedError('TODO: implement make_transform_pair using the stated contract')
    return train_transform, evaluation_transform




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def make_transform_pair(size: int) -&gt; Any:
    &quot;&quot;&quot;Input: square image size. Return separate train/evaluation transform objects. This first
    scratch baseline is deterministic; an augmentation experiment changes only the training
    transform.
    &quot;&quot;&quot;
    train_transform = build_image_transforms(size=size, training=False)
    evaluation_transform = build_image_transforms(size=size, training=False)
    return train_transform, evaluation_transform


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_train_transform, toy_eval_transform = make_transform_pair(32)
toy_image = Image.fromarray(np.zeros((12, 20, 3), dtype=np.uint8))
toy_pixels = toy_eval_transform(toy_image)
assert toy_pixels.shape == (3, 32, 32) and torch.isfinite(toy_pixels).all()
torch.testing.assert_close(toy_eval_transform(toy_image), toy_eval_transform(toy_image))
print("Transform checks passed: CHW output and deterministic evaluation.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_train_transform, toy_eval_transform = make_transform_pair(32)
toy_image = Image.fromarray(np.zeros((12, 20, 3), dtype=np.uint8))
toy_pixels = toy_eval_transform(toy_image)
assert toy_pixels.shape == (3, 32, 32) and torch.isfinite(toy_pixels).all()
torch.testing.assert_close(toy_eval_transform(toy_image), toy_eval_transform(toy_image))
print(&quot;Transform checks passed: CHW output and deterministic evaluation.&quot;)

</code></pre>
</details>


In [ ]:
train_transform, evaluation_transform = make_transform_pair(IMAGE_SIZE)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>train_transform, evaluation_transform = make_transform_pair(IMAGE_SIZE)

</code></pre>
</details>


### Worked component — `loader_for`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def loader_for(frame, training=False):
    dataset = ImageTableDataset(
        frame,
        image_column="image",
        target_column="label" if "label" in frame else None,
        label_to_index=mapping,
        root=DATA_DIR,
        transform=train_transform if training else evaluation_transform,
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=training, num_workers=0)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def loader_for(frame, training=False):
    dataset = ImageTableDataset(
        frame,
        image_column=&quot;image&quot;,
        target_column=&quot;label&quot; if &quot;label&quot; in frame else None,
        label_to_index=mapping,
        root=DATA_DIR,
        transform=train_transform if training else evaluation_transform,
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=training, num_workers=0)


</code></pre>
</details>


In [ ]:
train_loader, valid_loader = loader_for(tr, True), loader_for(va)
pixels, target = next(iter(train_loader))
assert pixels.ndim == 4 and target.dtype == torch.long
print("Batch image/label shapes:", pixels.shape, target.shape)
save_json(labels, OUTPUT / "labels.json")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>train_loader, valid_loader = loader_for(tr, True), loader_for(va)
pixels, target = next(iter(train_loader))
assert pixels.ndim == 4 and target.dtype == torch.long
print(&quot;Batch image/label shapes:&quot;, pixels.shape, target.shape)
save_json(labels, OUTPUT / &quot;labels.json&quot;)

</code></pre>
</details>


### Stop and explain

State the feature/batch shapes and the exact point at which preprocessing is fitted. What
happens to a previously unseen validation input?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

The scratch baseline uses deterministic resizing for its first sanity run. Training and
evaluation transforms are separate objects even when initially identical. Inspect a [B,3,H,W]
batch and its integer class indexes.

</details>



## 7. Connect outputs, loss and official metric

A compact CNN stacks convolutions, nonlinearities, pooling and a C-logit head. Adaptive pooling
removes dependence on the exact intermediate grid size. CrossEntropyLoss expects logits, without
an extra softmax.



### Supplied helper — `save_torch_state`

Atomically save torch state without importing torch until this function is used.




In [ ]:
def save_torch_state(value: Any, path: Path | str) -> Path:
    """Atomically save torch state without importing torch until this function is used."""
    return _write(value, path, "torch")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def save_torch_state(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save torch state without importing torch until this function is used.&quot;&quot;&quot;
    return _write(value, path, &quot;torch&quot;)


</code></pre>
</details>


### Supplied helper — `sync_artifacts`

Copy the entire candidate, including encoders, preserving separate run folders.




In [ ]:
def sync_artifacts(source: Path | str, destination: Path | str | None) -> list[Path]:
    """Copy the entire candidate, including encoders, preserving separate run folders."""
    if destination is None:
        return []
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if (
        destination == source
        or destination.is_relative_to(source)
        or source.is_relative_to(destination)
    ):
        raise ValueError("Output and backup folders must be separate")
    copied = []
    for path in source.rglob("*"):
        if path.is_file() and not path.name.endswith(".tmp"):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(f".{target.name}.{uuid4().hex}.tmp")
            try:
                shutil.copy2(path, temporary)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            copied.append(target)
    return copied




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def sync_artifacts(source: Path | str, destination: Path | str | None) -&gt; list[Path]:
    &quot;&quot;&quot;Copy the entire candidate, including encoders, preserving separate run folders.&quot;&quot;&quot;
    if destination is None:
        return []
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if (
        destination == source
        or destination.is_relative_to(source)
        or source.is_relative_to(destination)
    ):
        raise ValueError(&quot;Output and backup folders must be separate&quot;)
    copied = []
    for path in source.rglob(&quot;*&quot;):
        if path.is_file() and not path.name.endswith(&quot;.tmp&quot;):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(f&quot;.{target.name}.{uuid4().hex}.tmp&quot;)
            try:
                shutil.copy2(path, temporary)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            copied.append(target)
    return copied


</code></pre>
</details>


### Worked component — `evaluate_metric`

Only the metrics used in this lesson; task-specific graders appear beside the model.




In [ ]:
def evaluate_metric(name: str, truth: Any, prediction: Any, **kwargs: Any) -> float:
    """Only the metrics used in this lesson; task-specific graders appear beside the model."""
    from sklearn.metrics import accuracy_score, roc_auc_score

    if name == "roc_auc":
        return float(roc_auc_score(truth, prediction, **kwargs))
    if name == "accuracy":
        return float(accuracy_score(truth, prediction))
    if name == "dice_mean_image":
        y, p = np.asarray(truth, bool), np.asarray(prediction, bool)
        if y.shape != p.shape or not y.size:
            raise ValueError("Need aligned nonempty masks")
        axes = tuple(range(1, y.ndim))
        denominator = y.sum(axis=axes) + p.sum(axis=axes)
        return float(
            np.divide(
                2 * (y & p).sum(axis=axes), denominator, out=np.ones(len(y)), where=denominator != 0
            ).mean()
        )
    raise ValueError(f"Unsupported lesson metric: {name}")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def evaluate_metric(name: str, truth: Any, prediction: Any, **kwargs: Any) -&gt; float:
    &quot;&quot;&quot;Only the metrics used in this lesson; task-specific graders appear beside the model.&quot;&quot;&quot;
    from sklearn.metrics import accuracy_score, roc_auc_score

    if name == &quot;roc_auc&quot;:
        return float(roc_auc_score(truth, prediction, **kwargs))
    if name == &quot;accuracy&quot;:
        return float(accuracy_score(truth, prediction))
    if name == &quot;dice_mean_image&quot;:
        y, p = np.asarray(truth, bool), np.asarray(prediction, bool)
        if y.shape != p.shape or not y.size:
            raise ValueError(&quot;Need aligned nonempty masks&quot;)
        axes = tuple(range(1, y.ndim))
        denominator = y.sum(axis=axes) + p.sum(axis=axes)
        return float(
            np.divide(
                2 * (y &amp; p).sum(axis=axes), denominator, out=np.ones(len(y)), where=denominator != 0
            ).mean()
        )
    raise ValueError(f&quot;Unsupported lesson metric: {name}&quot;)


</code></pre>
</details>


### Exercise 4 — `make_model`

Construct the stated scratch architecture with the exact output dimensions and legal weight
policy. The actual batch/model compatibility check below must pass before any full training.

<details>
<summary>Hint 1 — choose the operation</summary>

A compact CNN stacks convolutions, nonlinearities, pooling and a C-logit head. Adaptive pooling
removes dependence on the exact intermediate grid size. CrossEntropyLoss expects logits, without
an extra softmax.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

end a compact scratch CNN with one raw logit per class

</details>




In [ ]:
def make_model(num_classes: int | None = None) -> nn.Module:
    """Build a scratch CNN from RGB images [B,3,H,W] to raw logits [B,C]."""
    classes = len(labels) if num_classes is None else num_classes
    raise NotImplementedError('TODO: end a compact scratch CNN with one raw logit per class')




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def make_model(num_classes: int | None = None) -&gt; nn.Module:
    &quot;&quot;&quot;Build a scratch CNN from RGB images [B,3,H,W] to raw logits [B,C].&quot;&quot;&quot;
    classes = len(labels) if num_classes is None else num_classes
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, classes),
    )


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
with torch.random.fork_rng(devices=[]), torch.no_grad():
    toy_network = make_model(num_classes=3).eval()
    toy_logits = toy_network(torch.zeros(2, 3, 24, 32))
    assert toy_logits.shape == (2, 3) and torch.isfinite(toy_logits).all()
    toy_loss = nn.CrossEntropyLoss()(toy_logits, torch.tensor([0, 2]))
    assert torch.isfinite(toy_loss)
print("CNN checks passed: configurable class count, rectangular inputs and compatible loss.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>with torch.random.fork_rng(devices=[]), torch.no_grad():
    toy_network = make_model(num_classes=3).eval()
    toy_logits = toy_network(torch.zeros(2, 3, 24, 32))
    assert toy_logits.shape == (2, 3) and torch.isfinite(toy_logits).all()
    toy_loss = nn.CrossEntropyLoss()(toy_logits, torch.tensor([0, 2]))
    assert torch.isfinite(toy_loss)
print(&quot;CNN checks passed: configurable class count, rectangular inputs and compatible loss.&quot;)

</code></pre>
</details>


In [ ]:
model = make_model()
criterion = nn.CrossEntropyLoss(weight=weights if USE_IMBALANCE_WEIGHTS else None).to(
    competition.device
)
MODEL_DESCRIPTION = "two-block scratch CNN"



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>model = make_model()
criterion = nn.CrossEntropyLoss(weight=weights if USE_IMBALANCE_WEIGHTS else None).to(
    competition.device
)
MODEL_DESCRIPTION = &quot;two-block scratch CNN&quot;

</code></pre>
</details>


### Exercise 5 — `score_predictions`

Input: aligned original validation labels [N] and decoded predicted labels [N]. Return the
fraction correctly classified. Macro F1 remains a separate diagnostic.

<details>
<summary>Hint 1 — choose the operation</summary>

Select using classification accuracy. Decode through the saved class order, inspect a confusion
matrix and difficult images, and report macro F1 for minority-class diagnosis without changing
the selection metric.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Use the task-specific evaluator without thresholding probability metrics or changing the target
scale.

</details>




In [ ]:
def score_predictions(truth: np.ndarray, predictions: np.ndarray) -> Any:
    """Input: aligned original validation labels [N] and decoded predicted labels [N]. Return
    the fraction correctly classified. Macro F1 remains a separate diagnostic.
    """
    raise NotImplementedError('TODO: implement score_predictions using the stated contract')
    return validation_score




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def score_predictions(truth: np.ndarray, predictions: np.ndarray) -&gt; Any:
    &quot;&quot;&quot;Input: aligned original validation labels [N] and decoded predicted labels [N]. Return
    the fraction correctly classified. Macro F1 remains a separate diagnostic.
    &quot;&quot;&quot;
    validation_score = evaluate_metric(&quot;accuracy&quot;, truth, predictions)
    return validation_score


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
np.testing.assert_allclose(score_predictions(np.array([0, 0, 1]), np.array([0, 1, 1])), 2 / 3)
print("Accuracy check passed: two of three decoded labels are correct.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>np.testing.assert_allclose(score_predictions(np.array([0, 0, 1]), np.array([0, 1, 1])), 2 / 3)
print(&quot;Accuracy check passed: two of three decoded labels are correct.&quot;)

</code></pre>
</details>


**Actual model/batch compatibility.** Run an evaluation forward pass before full training.
Expected: finite logits with the stated batch/output dimensions. Restore the model's previous
mode afterward.



In [ ]:
was_training = model.training
model.eval()
with torch.no_grad():
    example_inputs, example_targets = next(iter(valid_loader))
    example_inputs = example_inputs[:2].to(next(model.parameters()).device)
    example_logits = model(example_inputs)
    assert tuple(example_logits.shape) == (len(example_inputs), len(labels)), (
        "Model outputs do not match the task contract"
    )
    assert torch.isfinite(example_logits).all()
model.train(was_training)
print("Actual batch/model compatibility passed:", tuple(example_logits.shape))



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>was_training = model.training
model.eval()
with torch.no_grad():
    example_inputs, example_targets = next(iter(valid_loader))
    example_inputs = example_inputs[:2].to(next(model.parameters()).device)
    example_logits = model(example_inputs)
    assert tuple(example_logits.shape) == (len(example_inputs), len(labels)), (
        &quot;Model outputs do not match the task contract&quot;
    )
    assert torch.isfinite(example_logits).all()
model.train(was_training)
print(&quot;Actual batch/model compatibility passed:&quot;, tuple(example_logits.shape))

</code></pre>
</details>


### Stop and explain

Explain the output shape, loss and prediction decoding without reading the implementation. Which
convention would silently produce an invalid score?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

A compact CNN stacks convolutions, nonlinearities, pooling and a C-logit head. Adaptive pooling
removes dependence on the exact intermediate grid size. CrossEntropyLoss expects logits, without
an extra softmax. Select using classification accuracy. Decode through the saved class order,
inspect a confusion matrix and difficult images, and report macro F1 for minority-class
diagnosis without changing the selection metric.

</details>



## 8. Train and preserve a recoverable candidate

First implement one optimizer update on a separate tiny network: train mode, clear gradients,
forward, loss, backward, step. Then use the supplied hardware-aware loop for the real candidate
and reload its best checkpoint.



### Supplied helper — `TrainerConfig`

Small lesson loop: AdamW, one step per batch, no scheduler.




In [ ]:
@dataclass
class TrainerConfig:
    """Small lesson loop: AdamW, one step per batch, no scheduler."""

    epochs: int = 3
    learning_rate: float = 1e-3
    metric_name: str = "accuracy"
    mixed_precision: str = "auto"
    scheduler: str = "none"
    patience: int = 2




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass
class TrainerConfig:
    &quot;&quot;&quot;Small lesson loop: AdamW, one step per batch, no scheduler.&quot;&quot;&quot;

    epochs: int = 3
    learning_rate: float = 1e-3
    metric_name: str = &quot;accuracy&quot;
    mixed_precision: str = &quot;auto&quot;
    scheduler: str = &quot;none&quot;
    patience: int = 2


</code></pre>
</details>


### Supplied helper — `Trainer`

Visible training loop: validate globally, keep the best, reload before returning.




In [ ]:
class Trainer:
    """Visible training loop: validate globally, keep the best, reload before returning."""

    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        config: TrainerConfig,
        *,
        output_dir: Path,
        device: str,
        metric_fn: Callable | None = None,
        prediction_decoder: Callable | None = None,
        time_budget: TimeBudget | None = None,
        backup_dir: Path | None = None,
    ) -> None:
        if config.scheduler != "none" or config.mixed_precision not in {"auto", "none"}:
            raise ValueError("This minimal loop supports scheduler=none and AMP auto/none")
        if config.epochs < 1 or config.learning_rate <= 0 or config.patience < 1:
            raise ValueError("Epochs, learning rate and patience must be positive")
        self.model, self.criterion = model.to(device), criterion.to(device)
        self.config, self.device = config, device
        self.metric_fn, self.decoder = metric_fn, prediction_decoder
        self.budget, self.backup_dir = time_budget, backup_dir
        self.output_dir = Path(output_dir)
        self.checkpoint_path = self.output_dir / "best.pt"
        self.amp = torch.device(device).type == "cuda" and config.mixed_precision == "auto"
        self.dtype = (
            torch.bfloat16 if self.amp and torch.cuda.is_bf16_supported() else torch.float16
        )
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.amp and self.dtype == torch.float16)
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate, weight_decay=1e-2
        )

    @torch.inference_mode()
    def evaluate(self, loader: Iterable) -> dict[str, float]:
        self.model.eval()
        truths, predictions, total, count = [], [], 0.0, 0
        for inputs, targets in loader:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            with torch.autocast(torch.device(self.device).type, dtype=self.dtype, enabled=self.amp):
                logits = self.model(inputs)
                loss = self.criterion(logits, targets)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite validation loss")
            decoded = self.decoder(logits.float()) if self.decoder else logits.argmax(1)
            truths.append(targets.cpu().numpy())
            predictions.append(decoded.cpu().numpy())
            total += float(loss) * len(targets)
            count += len(targets)
        if not count:
            raise ValueError("Empty validation fold")
        # Salt overrides this method to select by the exact original-resolution grader.
        score = (
            self.metric_fn(np.concatenate(truths), np.concatenate(predictions))
            if self.metric_fn
            else -total / count
        )
        return {"loss": total / count, "metric": float(score)}

    def fit(self, train_loader: Iterable, valid_loader: Iterable) -> list[dict]:
        history, best, stale = [], -float("inf"), 0
        for epoch in range(self.config.epochs):
            if self.budget and self.budget.should_stop:
                break
            self.model.train()
            total, count = 0.0, 0
            for inputs, targets in train_loader:
                if self.budget and self.budget.should_stop:
                    break
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                self.optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    torch.device(self.device).type, dtype=self.dtype, enabled=self.amp
                ):
                    loss = self.criterion(self.model(inputs), targets)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite training loss")
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                total += float(loss.detach()) * len(targets)
                count += len(targets)
            if not count or (self.budget and self.budget.should_stop):
                break
            values = self.evaluate(valid_loader)
            if not np.isfinite(values["metric"]):
                raise ValueError("Nonfinite selection metric")
            history.append({"epoch": epoch, "train_loss": total / count, **values})
            print(history[-1])
            with (self.output_dir / "history.jsonl").open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(history[-1], allow_nan=False) + "\n")
            if values["metric"] > best:
                best, stale = values["metric"], 0
                save_torch_state(
                    {"model": self.model.state_dict(), "epoch": epoch, "score": best},
                    self.checkpoint_path,
                )
                sync_artifacts(self.output_dir, self.backup_dir)
            else:
                stale += 1
            if stale >= self.config.patience:
                break
        if not self.checkpoint_path.is_file():
            raise TimeoutError("No complete validation/checkpoint; shorten the debug candidate")
        self.model.load_state_dict(
            torch.load(self.checkpoint_path, map_location=self.device, weights_only=True)["model"]
        )
        return history




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>class Trainer:
    &quot;&quot;&quot;Visible training loop: validate globally, keep the best, reload before returning.&quot;&quot;&quot;

    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        config: TrainerConfig,
        *,
        output_dir: Path,
        device: str,
        metric_fn: Callable | None = None,
        prediction_decoder: Callable | None = None,
        time_budget: TimeBudget | None = None,
        backup_dir: Path | None = None,
    ) -&gt; None:
        if config.scheduler != &quot;none&quot; or config.mixed_precision not in {&quot;auto&quot;, &quot;none&quot;}:
            raise ValueError(&quot;This minimal loop supports scheduler=none and AMP auto/none&quot;)
        if config.epochs &lt; 1 or config.learning_rate &lt;= 0 or config.patience &lt; 1:
            raise ValueError(&quot;Epochs, learning rate and patience must be positive&quot;)
        self.model, self.criterion = model.to(device), criterion.to(device)
        self.config, self.device = config, device
        self.metric_fn, self.decoder = metric_fn, prediction_decoder
        self.budget, self.backup_dir = time_budget, backup_dir
        self.output_dir = Path(output_dir)
        self.checkpoint_path = self.output_dir / &quot;best.pt&quot;
        self.amp = torch.device(device).type == &quot;cuda&quot; and config.mixed_precision == &quot;auto&quot;
        self.dtype = (
            torch.bfloat16 if self.amp and torch.cuda.is_bf16_supported() else torch.float16
        )
        self.scaler = torch.amp.GradScaler(&quot;cuda&quot;, enabled=self.amp and self.dtype == torch.float16)
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate, weight_decay=1e-2
        )

    @torch.inference_mode()
    def evaluate(self, loader: Iterable) -&gt; dict[str, float]:
        self.model.eval()
        truths, predictions, total, count = [], [], 0.0, 0
        for inputs, targets in loader:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            with torch.autocast(torch.device(self.device).type, dtype=self.dtype, enabled=self.amp):
                logits = self.model(inputs)
                loss = self.criterion(logits, targets)
            if not torch.isfinite(loss):
                raise ValueError(&quot;Nonfinite validation loss&quot;)
            decoded = self.decoder(logits.float()) if self.decoder else logits.argmax(1)
            truths.append(targets.cpu().numpy())
            predictions.append(decoded.cpu().numpy())
            total += float(loss) * len(targets)
            count += len(targets)
        if not count:
            raise ValueError(&quot;Empty validation fold&quot;)
        # Salt overrides this method to select by the exact original-resolution grader.
        score = (
            self.metric_fn(np.concatenate(truths), np.concatenate(predictions))
            if self.metric_fn
            else -total / count
        )
        return {&quot;loss&quot;: total / count, &quot;metric&quot;: float(score)}

    def fit(self, train_loader: Iterable, valid_loader: Iterable) -&gt; list[dict]:
        history, best, stale = [], -float(&quot;inf&quot;), 0
        for epoch in range(self.config.epochs):
            if self.budget and self.budget.should_stop:
                break
            self.model.train()
            total, count = 0.0, 0
            for inputs, targets in train_loader:
                if self.budget and self.budget.should_stop:
                    break
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                self.optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    torch.device(self.device).type, dtype=self.dtype, enabled=self.amp
                ):
                    loss = self.criterion(self.model(inputs), targets)
                if not torch.isfinite(loss):
                    raise ValueError(&quot;Nonfinite training loss&quot;)
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                total += float(loss.detach()) * len(targets)
                count += len(targets)
            if not count or (self.budget and self.budget.should_stop):
                break
            values = self.evaluate(valid_loader)
            if not np.isfinite(values[&quot;metric&quot;]):
                raise ValueError(&quot;Nonfinite selection metric&quot;)
            history.append({&quot;epoch&quot;: epoch, &quot;train_loss&quot;: total / count, **values})
            print(history[-1])
            with (self.output_dir / &quot;history.jsonl&quot;).open(&quot;a&quot;, encoding=&quot;utf-8&quot;) as handle:
                handle.write(json.dumps(history[-1], allow_nan=False) + &quot;\n&quot;)
            if values[&quot;metric&quot;] &gt; best:
                best, stale = values[&quot;metric&quot;], 0
                save_torch_state(
                    {&quot;model&quot;: self.model.state_dict(), &quot;epoch&quot;: epoch, &quot;score&quot;: best},
                    self.checkpoint_path,
                )
                sync_artifacts(self.output_dir, self.backup_dir)
            else:
                stale += 1
            if stale &gt;= self.config.patience:
                break
        if not self.checkpoint_path.is_file():
            raise TimeoutError(&quot;No complete validation/checkpoint; shorten the debug candidate&quot;)
        self.model.load_state_dict(
            torch.load(self.checkpoint_path, map_location=self.device, weights_only=True)[&quot;model&quot;]
        )
        return history


</code></pre>
</details>


### Exercise 6 — `one_batch_step`

Input: a network, (pixels, integer labels) batch, optimizer, criterion and device. Return one
finite scalar loss after updating parameters. Do not mutate the real candidate during this tiny
drill.

<details>
<summary>Hint 1 — choose the operation</summary>

First implement one optimizer update on a separate tiny network: train mode, clear gradients,
forward, loss, backward, step. Then use the supplied hardware-aware loop for the real candidate
and reload its best checkpoint.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

reconstruct the essential PyTorch optimizer update

</details>




In [ ]:
def one_batch_step(network, batch, optimizer, loss_function, device):
    network.train()
    pixels, targets = (value.to(device) for value in batch)
    raise NotImplementedError('TODO: reconstruct the essential PyTorch optimizer update')
    return float(loss.detach())




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def one_batch_step(network, batch, optimizer, loss_function, device):
    network.train()
    pixels, targets = (value.to(device) for value in batch)
    optimizer.zero_grad(set_to_none=True)
    loss = loss_function(network(pixels), targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
with torch.random.fork_rng(devices=[]):
    toy_network = nn.Linear(2, 2)
    toy_optimizer = torch.optim.SGD(toy_network.parameters(), lr=0.1)
    toy_before = toy_network.weight.detach().clone()
    toy_batch = (torch.ones(4, 2), torch.zeros(4, dtype=torch.long))
    toy_loss = one_batch_step(toy_network, toy_batch, toy_optimizer, nn.CrossEntropyLoss(), "cpu")
    assert np.isfinite(toy_loss), "One update must return a finite scalar loss"
    assert not torch.equal(toy_before, toy_network.weight.detach()), "No optimizer update occurred"
    assert toy_network.training, "The update must enable train mode"
print("Optimizer checks passed on a separate tiny CPU model; the real candidate is untouched.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>with torch.random.fork_rng(devices=[]):
    toy_network = nn.Linear(2, 2)
    toy_optimizer = torch.optim.SGD(toy_network.parameters(), lr=0.1)
    toy_before = toy_network.weight.detach().clone()
    toy_batch = (torch.ones(4, 2), torch.zeros(4, dtype=torch.long))
    toy_loss = one_batch_step(toy_network, toy_batch, toy_optimizer, nn.CrossEntropyLoss(), &quot;cpu&quot;)
    assert np.isfinite(toy_loss), &quot;One update must return a finite scalar loss&quot;
    assert not torch.equal(toy_before, toy_network.weight.detach()), &quot;No optimizer update occurred&quot;
    assert toy_network.training, &quot;The update must enable train mode&quot;
print(&quot;Optimizer checks passed on a separate tiny CPU model; the real candidate is untouched.&quot;)

</code></pre>
</details>


In [ ]:
started = perf_counter()
training = TrainerConfig(
    epochs=EPOCHS,
    learning_rate=1e-3,
    metric_name="accuracy",
    scheduler="none",
    mixed_precision="auto",
    patience=2,
)
trainer = Trainer(
    model,
    criterion,
    training,
    output_dir=OUTPUT,
    device=competition.device,
    metric_fn=lambda y, p: evaluate_metric("accuracy", y, p),
    time_budget=budget,
    backup_dir=DURABLE_OUTPUT,
)
history = trainer.fit(train_loader, valid_loader)
fit_seconds = perf_counter() - started
checkpoint_path = trainer.checkpoint_path
validation_logits = predict_logits(model, valid_loader, device=competition.device)
validation_predictions = validation_logits.argmax(1)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>started = perf_counter()
training = TrainerConfig(
    epochs=EPOCHS,
    learning_rate=1e-3,
    metric_name=&quot;accuracy&quot;,
    scheduler=&quot;none&quot;,
    mixed_precision=&quot;auto&quot;,
    patience=2,
)
trainer = Trainer(
    model,
    criterion,
    training,
    output_dir=OUTPUT,
    device=competition.device,
    metric_fn=lambda y, p: evaluate_metric(&quot;accuracy&quot;, y, p),
    time_budget=budget,
    backup_dir=DURABLE_OUTPUT,
)
history = trainer.fit(train_loader, valid_loader)
fit_seconds = perf_counter() - started
checkpoint_path = trainer.checkpoint_path
validation_logits = predict_logits(model, valid_loader, device=competition.device)
validation_predictions = validation_logits.argmax(1)

</code></pre>
</details>


In [ ]:
selected_threshold = 0.5



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>selected_threshold = 0.5

</code></pre>
</details>


In [ ]:
assert checkpoint_path.is_file(), "No saved candidate is available for inference"
assert np.isfinite(fit_seconds) and fit_seconds >= 0
print("Candidate checks passed: persisted artifact and measured fit time.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert checkpoint_path.is_file(), &quot;No saved candidate is available for inference&quot;
assert np.isfinite(fit_seconds) and fit_seconds &gt;= 0
print(&quot;Candidate checks passed: persisted artifact and measured fit time.&quot;)

</code></pre>
</details>


### Stop and explain

What evidence proves the saved artifact can reproduce inference? Which training state is not
promised by this reload?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Keep architecture/preprocessing, label order and best weights together, then compare reloaded
validation predictions. This lesson preserves inference recovery, not an exact continuation of
optimizer/scaler/random state.

</details>



## 9. Interpret validation and compare experiments

Select using classification accuracy. Decode through the saved class order, inspect a confusion
matrix and difficult images, and report macro F1 for minority-class diagnosis without changing
the selection metric.



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

truth = va["label"].map(mapping).to_numpy()
validation_score = score_predictions(truth, validation_predictions)
print("Official accuracy:", validation_score)
print("Training-majority reference:", (va["label"] == tr["label"].mode().iloc[0]).mean())
print(
    classification_report(
        truth,
        validation_predictions,
        labels=list(range(len(labels))),
        target_names=[str(value) for value in labels],
        zero_division=0,
    )
)
errors = va[[ID_COLUMN, "label", "image"]].copy()
errors["prediction"] = np.asarray(labels)[validation_predictions]
print(errors.loc[errors["label"] != errors["prediction"]].head(20).to_string(index=False))
errors.to_csv(OUTPUT / "validation_errors.csv", index=False)
plt.imshow(
    confusion_matrix(truth, validation_predictions, labels=list(range(len(labels)))), cmap="Blues"
)
plt.xlabel("Predicted class index")
plt.ylabel("True class index")
plt.savefig(OUTPUT / "validation.png")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from sklearn.metrics import classification_report, confusion_matrix

truth = va[&quot;label&quot;].map(mapping).to_numpy()
validation_score = score_predictions(truth, validation_predictions)
print(&quot;Official accuracy:&quot;, validation_score)
print(&quot;Training-majority reference:&quot;, (va[&quot;label&quot;] == tr[&quot;label&quot;].mode().iloc[0]).mean())
print(
    classification_report(
        truth,
        validation_predictions,
        labels=list(range(len(labels))),
        target_names=[str(value) for value in labels],
        zero_division=0,
    )
)
errors = va[[ID_COLUMN, &quot;label&quot;, &quot;image&quot;]].copy()
errors[&quot;prediction&quot;] = np.asarray(labels)[validation_predictions]
print(errors.loc[errors[&quot;label&quot;] != errors[&quot;prediction&quot;]].head(20).to_string(index=False))
errors.to_csv(OUTPUT / &quot;validation_errors.csv&quot;, index=False)
plt.imshow(
    confusion_matrix(truth, validation_predictions, labels=list(range(len(labels)))), cmap=&quot;Blues&quot;
)
plt.xlabel(&quot;Predicted class index&quot;)
plt.ylabel(&quot;True class index&quot;)
plt.savefig(OUTPUT / &quot;validation.png&quot;)

</code></pre>
</details>


In [ ]:
plt.show()
plt.close()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>plt.show()
plt.close()

</code></pre>
</details>


In [ ]:
assert np.isfinite(validation_score), "Selection requires a finite official score"
print("Validation check passed. Compare this score and runtime with the baseline ledger.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert np.isfinite(validation_score), &quot;Selection requires a finite official score&quot;
print(&quot;Validation check passed. Compare this score and runtime with the baseline ledger.&quot;)

</code></pre>
</details>


### Stop and explain

What is the next single experiment, and what result would make you reject it?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Compare the scratch CNN with EfficientNet at a measured small resolution first. Separately test
augmentation and legally permitted initialization. Keep specimen groups fixed and include full
image inference cost.

</details>



## 10. Reload and infer every test item

Restore the class mapping and your best state before test inference. Validation logits must
agree within floating-point tolerance. Deterministic evaluation transforms and stable image
ordering are part of the saved model.



In [ ]:
started = perf_counter()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>started = perf_counter()

</code></pre>
</details>


### Exercise 7 — Reload and predict

Restore the class mapping and your best state before test inference. Validation logits must
agree within floating-point tolerance. Deterministic evaluation transforms and stable image
ordering are part of the saved model.

<details>
<summary>Hint 1 — choose the operation</summary>

Restore the class mapping and your best state before test inference. Validation logits must
agree within floating-point tolerance. Deterministic evaluation transforms and stable image
ordering are part of the saved model.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

recreate the class order and load your own best checkpoint without original weights

</details>



In [ ]:
raise NotImplementedError('TODO: recreate the class order and load your own best checkpoint without original weights')



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>labels = json.loads((OUTPUT / &quot;labels.json&quot;).read_text(encoding=&quot;utf-8&quot;))
mapping = {label: index for index, label in enumerate(labels)}
restored = make_model().to(competition.device)
restored.load_state_dict(
    torch.load(checkpoint_path, map_location=competition.device, weights_only=True)[&quot;model&quot;],
    strict=True,
)
np.testing.assert_allclose(
    predict_logits(restored, loader_for(va), device=competition.device),
    validation_logits,
    rtol=1e-5,
    atol=1e-6,
)
test_predictions = np.asarray(labels)[
    predict_logits(restored, loader_for(test), device=competition.device).argmax(1)
]

</code></pre>
</details>


In [ ]:
assert set(test_predictions) <= set(labels)
output_values = {"label": test_predictions}
save_json(
    {
        "labels": labels,
        "image_size": IMAGE_SIZE,
        "normalization": "ImageNet mean/std",
        "model_name": MODEL_NAME,
    },
    OUTPUT / "recipe.json",
)
inference_seconds = perf_counter() - started



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert set(test_predictions) &lt;= set(labels)
output_values = {&quot;label&quot;: test_predictions}
save_json(
    {
        &quot;labels&quot;: labels,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;normalization&quot;: &quot;ImageNet mean/std&quot;,
        &quot;model_name&quot;: MODEL_NAME,
    },
    OUTPUT / &quot;recipe.json&quot;,
)
inference_seconds = perf_counter() - started

</code></pre>
</details>


In [ ]:
assert set(output_values) == set(sample.columns) - {ID_COLUMN}
assert all(len(values) == len(test) for values in output_values.values())
print("Inference checks passed: every test item has the required output columns.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert set(output_values) == set(sample.columns) - {ID_COLUMN}
assert all(len(values) == len(test) for values in output_values.values())
print(&quot;Inference checks passed: every test item has the required output columns.&quot;)

</code></pre>
</details>


### Stop and explain

Can this saved candidate handle every private input without refitting or downloading anything?
Which assets must travel with it?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Restore the class mapping and your best state before test inference. Validation logits must
agree within floating-point tolerance. Deterministic evaluation transforms and stable image
ordering are part of the saved model.

</details>



## 11. Deliver and verify the submission

One cassava image has one of five class labels. image_id links the image to a submission row.
Preserve the recorded label-to-index mapping; the model emits raw logits, while the submission
requires original labels.

Write the CSV, read it back, and verify IDs, columns, row count, missing values and allowed
prediction values before saving the complete recipe. A local check does not confirm organizer
upload acceptance.



### Supplied helper — `SubmissionValidation`

Validation errors and non-fatal warnings for a candidate submission.




In [ ]:
@dataclass(slots=True)
class SubmissionValidation:
    """Validation errors and non-fatal warnings for a candidate submission."""

    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    def raise_for_errors(self) -> None:
        """Raise one ``ValueError`` containing every validation error."""
        if self.errors:
            raise ValueError("Invalid submission: " + "; ".join(self.errors))




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass(slots=True)
class SubmissionValidation:
    &quot;&quot;&quot;Validation errors and non-fatal warnings for a candidate submission.&quot;&quot;&quot;

    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    def raise_for_errors(self) -&gt; None:
        &quot;&quot;&quot;Raise one ``ValueError`` containing every validation error.&quot;&quot;&quot;
        if self.errors:
            raise ValueError(&quot;Invalid submission: &quot; + &quot;; &quot;.join(self.errors))


</code></pre>
</details>


### Supplied helper — `build_submission`

Copy the sample and optionally align predictions using unique (possibly composite) IDs.

Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.




In [ ]:
def build_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    *,
    prediction_ids: pd.DataFrame | Sequence[object] | None = None,
    id_columns: str | Iterable[str] | None = None,
) -> pd.DataFrame:
    """Copy the sample and optionally align predictions using unique (possibly composite) IDs.

    Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
    Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.
    """
    submission = sample.copy()
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    order = None
    if prediction_ids is not None:
        if not ids:
            raise ValueError("id_columns is required with prediction_ids")
        if isinstance(prediction_ids, pd.DataFrame):
            keys = prediction_ids[ids].reset_index(drop=True)
        elif len(ids) == 1:
            keys = pd.DataFrame({ids[0]: list(prediction_ids)})
        else:
            raise ValueError("Composite prediction IDs must be a DataFrame")
        for name, frame in (("sample", sample), ("predictions", keys)):
            if (
                frame[ids].isna().any().any()
                or frame[ids].eq("").any().any()
                or frame.duplicated(ids).any()
            ):
                raise ValueError(f"{name} IDs must be nonmissing and unique across {ids}")
        expected = pd.MultiIndex.from_frame(sample[ids])
        actual = pd.MultiIndex.from_frame(keys)
        order = actual.get_indexer(expected)
        if len(actual) != len(expected) or (order < 0).any():
            raise ValueError("Prediction IDs must match the sample IDs exactly")
    for column, values in predictions.items():
        if column not in submission.columns:
            raise KeyError(f"Prediction column not present in sample submission: {column}")
        if len(values) != len(submission):
            raise ValueError(
                f"Column {column} has {len(values)} values; expected {len(submission)}"
            )
        if column in ids:
            raise ValueError(f"Cannot overwrite ID column: {column}")
        values = list(values)
        submission[column] = values if order is None else [values[index] for index in order]
    return submission




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def build_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    *,
    prediction_ids: pd.DataFrame | Sequence[object] | None = None,
    id_columns: str | Iterable[str] | None = None,
) -&gt; pd.DataFrame:
    &quot;&quot;&quot;Copy the sample and optionally align predictions using unique (possibly composite) IDs.

    Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
    Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.
    &quot;&quot;&quot;
    submission = sample.copy()
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    order = None
    if prediction_ids is not None:
        if not ids:
            raise ValueError(&quot;id_columns is required with prediction_ids&quot;)
        if isinstance(prediction_ids, pd.DataFrame):
            keys = prediction_ids[ids].reset_index(drop=True)
        elif len(ids) == 1:
            keys = pd.DataFrame({ids[0]: list(prediction_ids)})
        else:
            raise ValueError(&quot;Composite prediction IDs must be a DataFrame&quot;)
        for name, frame in ((&quot;sample&quot;, sample), (&quot;predictions&quot;, keys)):
            if (
                frame[ids].isna().any().any()
                or frame[ids].eq(&quot;&quot;).any().any()
                or frame.duplicated(ids).any()
            ):
                raise ValueError(f&quot;{name} IDs must be nonmissing and unique across {ids}&quot;)
        expected = pd.MultiIndex.from_frame(sample[ids])
        actual = pd.MultiIndex.from_frame(keys)
        order = actual.get_indexer(expected)
        if len(actual) != len(expected) or (order &lt; 0).any():
            raise ValueError(&quot;Prediction IDs must match the sample IDs exactly&quot;)
    for column, values in predictions.items():
        if column not in submission.columns:
            raise KeyError(f&quot;Prediction column not present in sample submission: {column}&quot;)
        if len(values) != len(submission):
            raise ValueError(
                f&quot;Column {column} has {len(values)} values; expected {len(submission)}&quot;
            )
        if column in ids:
            raise ValueError(f&quot;Cannot overwrite ID column: {column}&quot;)
        values = list(values)
        submission[column] = values if order is None else [values[index] for index in order]
    return submission


</code></pre>
</details>


### Supplied helper — `validate_submission`

Check schema, row count, ID order, missing values, and optional allowed labels.




In [ ]:
def validate_submission(
    submission: pd.DataFrame,
    sample: pd.DataFrame,
    *,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -> SubmissionValidation:
    """Check schema, row count, ID order, missing values, and optional allowed labels."""
    errors: list[str] = []
    warnings: list[str] = []
    if submission.columns.duplicated().any() or sample.columns.duplicated().any():
        errors.append("Submission and sample column names must be unique")
        return SubmissionValidation(valid=False, errors=errors)
    if list(submission.columns) != list(sample.columns):
        errors.append(
            f"Columns/order differ: got {list(submission.columns)}, expected {list(sample.columns)}"
        )
    if len(submission) != len(sample):
        errors.append(f"Row count differs: got {len(submission)}, expected {len(sample)}")
    if submission.isna().any().any():
        columns = submission.columns[submission.isna().any()].tolist()
        errors.append(f"Missing values in columns: {columns}")
    numeric = submission.select_dtypes(include="number")
    if not np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all():
        errors.append("Non-finite numeric values are not allowed")
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    for column in ids:
        if column not in submission or column not in sample:
            errors.append(f"Missing ID column: {column}")
        elif submission[column].isna().any() or submission[column].eq("").any():
            errors.append(f"Missing ID values in column: {column}")
        elif len(submission) == len(sample) and not submission[column].equals(sample[column]):
            errors.append(f"ID values/order changed in column: {column}")
    if ids and all(column in submission for column in ids) and submission.duplicated(ids).any():
        warnings.append(f"Duplicate ID keys across columns: {ids}")
    for column, labels in (allowed_labels or {}).items():
        if column not in submission:
            errors.append(f"Missing allowed-label column: {column}")
        else:
            invalid = set(submission[column].dropna().unique()) - set(labels)
            if invalid:
                errors.append(f"Unexpected labels in {column}: {sorted(map(str, invalid))[:10]}")
    return SubmissionValidation(valid=not errors, errors=errors, warnings=warnings)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def validate_submission(
    submission: pd.DataFrame,
    sample: pd.DataFrame,
    *,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -&gt; SubmissionValidation:
    &quot;&quot;&quot;Check schema, row count, ID order, missing values, and optional allowed labels.&quot;&quot;&quot;
    errors: list[str] = []
    warnings: list[str] = []
    if submission.columns.duplicated().any() or sample.columns.duplicated().any():
        errors.append(&quot;Submission and sample column names must be unique&quot;)
        return SubmissionValidation(valid=False, errors=errors)
    if list(submission.columns) != list(sample.columns):
        errors.append(
            f&quot;Columns/order differ: got {list(submission.columns)}, expected {list(sample.columns)}&quot;
        )
    if len(submission) != len(sample):
        errors.append(f&quot;Row count differs: got {len(submission)}, expected {len(sample)}&quot;)
    if submission.isna().any().any():
        columns = submission.columns[submission.isna().any()].tolist()
        errors.append(f&quot;Missing values in columns: {columns}&quot;)
    numeric = submission.select_dtypes(include=&quot;number&quot;)
    if not np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all():
        errors.append(&quot;Non-finite numeric values are not allowed&quot;)
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    for column in ids:
        if column not in submission or column not in sample:
            errors.append(f&quot;Missing ID column: {column}&quot;)
        elif submission[column].isna().any() or submission[column].eq(&quot;&quot;).any():
            errors.append(f&quot;Missing ID values in column: {column}&quot;)
        elif len(submission) == len(sample) and not submission[column].equals(sample[column]):
            errors.append(f&quot;ID values/order changed in column: {column}&quot;)
    if ids and all(column in submission for column in ids) and submission.duplicated(ids).any():
        warnings.append(f&quot;Duplicate ID keys across columns: {ids}&quot;)
    for column, labels in (allowed_labels or {}).items():
        if column not in submission:
            errors.append(f&quot;Missing allowed-label column: {column}&quot;)
        else:
            invalid = set(submission[column].dropna().unique()) - set(labels)
            if invalid:
                errors.append(f&quot;Unexpected labels in {column}: {sorted(map(str, invalid))[:10]}&quot;)
    return SubmissionValidation(valid=not errors, errors=errors, warnings=warnings)


</code></pre>
</details>


### Supplied helper — `write_submission`

Validate then atomically write a UTF-8 CSV submission.




In [ ]:
def write_submission(
    submission: pd.DataFrame,
    path: Path | str,
    *,
    sample: pd.DataFrame | None = None,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -> Path:
    """Validate then atomically write a UTF-8 CSV submission."""
    if sample is not None:
        validation = validate_submission(
            submission, sample, id_columns=id_columns, allowed_labels=allowed_labels
        )
        validation.raise_for_errors()
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.{uuid4().hex}.tmp")
    try:
        submission.to_csv(temporary, index=False, encoding="utf-8")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def write_submission(
    submission: pd.DataFrame,
    path: Path | str,
    *,
    sample: pd.DataFrame | None = None,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -&gt; Path:
    &quot;&quot;&quot;Validate then atomically write a UTF-8 CSV submission.&quot;&quot;&quot;
    if sample is not None:
        validation = validate_submission(
            submission, sample, id_columns=id_columns, allowed_labels=allowed_labels
        )
        validation.raise_for_errors()
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f&quot;.{destination.name}.{uuid4().hex}.tmp&quot;)
    try:
        submission.to_csv(temporary, index=False, encoding=&quot;utf-8&quot;)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


</code></pre>
</details>


### Supplied helper — `_mapping`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _mapping(value: Any) -> Mapping[str, Any]:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return value
    raise TypeError("Expected a dataclass or mapping")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _mapping(value: Any) -&gt; Mapping[str, Any]:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return value
    raise TypeError(&quot;Expected a dataclass or mapping&quot;)


</code></pre>
</details>


### Supplied helper — `technical_report`

Render the standard competition handoff report as Markdown text.




In [ ]:
def technical_report(
    *,
    task: str,
    model: str,
    data_summary: Mapping[str, Any],
    split_summary: Mapping[str, Any],
    training_config: Any,
    validation_metrics: Mapping[str, float],
    augmentations: list[str] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -> str:
    """Render the standard competition handoff report as Markdown text."""
    config = _mapping(training_config)
    lines = [
        "# Technical Report",
        "",
        f"- Task: {task}",
        f"- Model: {model}",
        f"- Checkpoint: {checkpoint or 'not recorded'}",
        "",
        "## Data and validation",
        "",
    ]
    lines.extend(f"- {key}: {value}" for key, value in data_summary.items())
    lines.extend(f"- Split {key}: {value}" for key, value in split_summary.items())
    lines.extend(["", "## Training", ""])
    lines.extend(f"- {key}: {value}" for key, value in config.items())
    lines.extend(["", "## Validation metrics", ""])
    lines.extend(f"- {key}: {value:.6f}" for key, value in validation_metrics.items())
    lines.extend(["", "## Augmentations", ""])
    lines.extend(f"- {item}" for item in (augmentations or ["None"]))
    if notes:
        lines.extend(["", "## Notes", ""])
        lines.extend(f"- {item}" for item in notes)
    return "\n".join(lines) + "\n"




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def technical_report(
    *,
    task: str,
    model: str,
    data_summary: Mapping[str, Any],
    split_summary: Mapping[str, Any],
    training_config: Any,
    validation_metrics: Mapping[str, float],
    augmentations: list[str] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -&gt; str:
    &quot;&quot;&quot;Render the standard competition handoff report as Markdown text.&quot;&quot;&quot;
    config = _mapping(training_config)
    lines = [
        &quot;# Technical Report&quot;,
        &quot;&quot;,
        f&quot;- Task: {task}&quot;,
        f&quot;- Model: {model}&quot;,
        f&quot;- Checkpoint: {checkpoint or &#x27;not recorded&#x27;}&quot;,
        &quot;&quot;,
        &quot;## Data and validation&quot;,
        &quot;&quot;,
    ]
    lines.extend(f&quot;- {key}: {value}&quot; for key, value in data_summary.items())
    lines.extend(f&quot;- Split {key}: {value}&quot; for key, value in split_summary.items())
    lines.extend([&quot;&quot;, &quot;## Training&quot;, &quot;&quot;])
    lines.extend(f&quot;- {key}: {value}&quot; for key, value in config.items())
    lines.extend([&quot;&quot;, &quot;## Validation metrics&quot;, &quot;&quot;])
    lines.extend(f&quot;- {key}: {value:.6f}&quot; for key, value in validation_metrics.items())
    lines.extend([&quot;&quot;, &quot;## Augmentations&quot;, &quot;&quot;])
    lines.extend(f&quot;- {item}&quot; for item in (augmentations or [&quot;None&quot;]))
    if notes:
        lines.extend([&quot;&quot;, &quot;## Notes&quot;, &quot;&quot;])
        lines.extend(f&quot;- {item}&quot; for item in notes)
    return &quot;\n&quot;.join(lines) + &quot;\n&quot;


</code></pre>
</details>


### Supplied helper — `write_technical_report`

Write a UTF-8 technical report, creating its parent directory.




In [ ]:
def write_technical_report(content: str, path: Path | str) -> Path:
    """Write a UTF-8 technical report, creating its parent directory."""
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def write_technical_report(content: str, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Write a UTF-8 technical report, creating its parent directory.&quot;&quot;&quot;
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding=&quot;utf-8&quot;)
    return destination


</code></pre>
</details>


### Supplied helper — `record_run`

Save configuration/fold positions and a matching human-readable technical report.




In [ ]:
def record_run(
    output_dir: Path | str,
    *,
    task: str,
    model: str,
    configuration: Any,
    validation_metrics: Mapping[str, float],
    split: SplitResult | None = None,
    data_summary: Mapping[str, Any] | None = None,
    inference_options: Mapping[str, Any] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -> Path:
    """Save configuration/fold positions and a matching human-readable technical report."""
    root = Path(output_dir)
    split_summary = (
        {"train_rows": len(split.train), "valid_rows": len(split.valid)} if split else {}
    )
    payload = {
        "task": task,
        "model": model,
        "configuration": _mapping(configuration),
        "validation_metrics": dict(validation_metrics),
        "data_summary": dict(data_summary or {}),
        "inference_options": dict(inference_options or {}),
        "checkpoint": checkpoint,
        "notes": notes or [],
        "split": {
            **split_summary,
            "train_indices": split.train_indices if split else [],
            "valid_indices": split.valid_indices if split else [],
        },
    }
    save_json(payload, root / "run.json")
    report = technical_report(
        task=task,
        model=model,
        data_summary=data_summary or {},
        split_summary=split_summary,
        training_config=configuration,
        validation_metrics=validation_metrics,
        checkpoint=checkpoint,
        notes=[*(notes or []), f"Inference options: {dict(inference_options or {})}"],
    )
    return write_technical_report(report, root / "technical_report.md")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def record_run(
    output_dir: Path | str,
    *,
    task: str,
    model: str,
    configuration: Any,
    validation_metrics: Mapping[str, float],
    split: SplitResult | None = None,
    data_summary: Mapping[str, Any] | None = None,
    inference_options: Mapping[str, Any] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -&gt; Path:
    &quot;&quot;&quot;Save configuration/fold positions and a matching human-readable technical report.&quot;&quot;&quot;
    root = Path(output_dir)
    split_summary = (
        {&quot;train_rows&quot;: len(split.train), &quot;valid_rows&quot;: len(split.valid)} if split else {}
    )
    payload = {
        &quot;task&quot;: task,
        &quot;model&quot;: model,
        &quot;configuration&quot;: _mapping(configuration),
        &quot;validation_metrics&quot;: dict(validation_metrics),
        &quot;data_summary&quot;: dict(data_summary or {}),
        &quot;inference_options&quot;: dict(inference_options or {}),
        &quot;checkpoint&quot;: checkpoint,
        &quot;notes&quot;: notes or [],
        &quot;split&quot;: {
            **split_summary,
            &quot;train_indices&quot;: split.train_indices if split else [],
            &quot;valid_indices&quot;: split.valid_indices if split else [],
        },
    }
    save_json(payload, root / &quot;run.json&quot;)
    report = technical_report(
        task=task,
        model=model,
        data_summary=data_summary or {},
        split_summary=split_summary,
        training_config=configuration,
        validation_metrics=validation_metrics,
        checkpoint=checkpoint,
        notes=[*(notes or []), f&quot;Inference options: {dict(inference_options or {})}&quot;],
    )
    return write_technical_report(report, root / &quot;technical_report.md&quot;)


</code></pre>
</details>


### Exercise 8 — `deliver_submission`

Input: sample template, named predictions, prediction IDs, key-column name and output path.
Return: ID-aligned checked frame and saved CSV path. Reject duplicated, missing or extra
prediction keys.

<details>
<summary>Hint 1 — choose the operation</summary>

One cassava image has one of five class labels. image_id links the image to a submission row.
Preserve the recorded label-to-index mapping; the model emits raw logits, while the submission
requires original labels.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Build by ID, raise validation errors, then write the validated frame. Never assume test order
equals sample order.

</details>




In [ ]:
def deliver_submission(
    sample_frame: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    prediction_ids: Sequence[object],
    id_column: str,
    output_path: Path,
) -> Any:
    """Input: sample template, named predictions, prediction IDs, key-column name and output
    path. Return: ID-aligned checked frame and saved CSV path. Reject duplicated, missing or
    extra prediction keys.
    """
    raise NotImplementedError('TODO: implement deliver_submission using the stated contract')
    return submission, destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def deliver_submission(
    sample_frame: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    prediction_ids: Sequence[object],
    id_column: str,
    output_path: Path,
) -&gt; Any:
    &quot;&quot;&quot;Input: sample template, named predictions, prediction IDs, key-column name and output
    path. Return: ID-aligned checked frame and saved CSV path. Reject duplicated, missing or
    extra prediction keys.
    &quot;&quot;&quot;
    submission = build_submission(
        sample_frame, predictions, prediction_ids=prediction_ids, id_columns=id_column
    )
    validate_submission(submission, sample_frame, id_columns=id_column).raise_for_errors()
    destination = write_submission(submission, output_path)
    return submission, destination


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_sample = pd.DataFrame({"key": ["001", "NA"], "prediction": [0.0, 0.0]})
toy_submission, toy_path = deliver_submission(
    toy_sample, {"prediction": [0.8, 0.2]}, ["NA", "001"], "key", OUTPUT / "checks" / "delivery.csv"
)
assert toy_submission["key"].tolist() == ["001", "NA"], "Preserve literal IDs and sample order"
np.testing.assert_allclose(toy_submission["prediction"], [0.2, 0.8])
toy_loaded = pd.read_csv(toy_path, converters={"key": str}, keep_default_na=False)
assert toy_loaded["key"].tolist() == ["001", "NA"]
try:
    deliver_submission(
        toy_sample,
        {"prediction": [0.2, 0.8]},
        ["001", "001"],
        "key",
        OUTPUT / "checks" / "invalid.csv",
    )
except ValueError:
    pass
else:
    raise AssertionError("Duplicate prediction IDs must be rejected, not guessed")
print("Delivery checks passed: shuffled/literal IDs and duplicate rejection.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_sample = pd.DataFrame({&quot;key&quot;: [&quot;001&quot;, &quot;NA&quot;], &quot;prediction&quot;: [0.0, 0.0]})
toy_submission, toy_path = deliver_submission(
    toy_sample, {&quot;prediction&quot;: [0.8, 0.2]}, [&quot;NA&quot;, &quot;001&quot;], &quot;key&quot;, OUTPUT / &quot;checks&quot; / &quot;delivery.csv&quot;
)
assert toy_submission[&quot;key&quot;].tolist() == [&quot;001&quot;, &quot;NA&quot;], &quot;Preserve literal IDs and sample order&quot;
np.testing.assert_allclose(toy_submission[&quot;prediction&quot;], [0.2, 0.8])
toy_loaded = pd.read_csv(toy_path, converters={&quot;key&quot;: str}, keep_default_na=False)
assert toy_loaded[&quot;key&quot;].tolist() == [&quot;001&quot;, &quot;NA&quot;]
try:
    deliver_submission(
        toy_sample,
        {&quot;prediction&quot;: [0.2, 0.8]},
        [&quot;001&quot;, &quot;001&quot;],
        &quot;key&quot;,
        OUTPUT / &quot;checks&quot; / &quot;invalid.csv&quot;,
    )
except ValueError:
    pass
else:
    raise AssertionError(&quot;Duplicate prediction IDs must be rejected, not guessed&quot;)
print(&quot;Delivery checks passed: shuffled/literal IDs and duplicate rejection.&quot;)

</code></pre>
</details>


In [ ]:
submission, destination = deliver_submission(
    sample, output_values, test[ID_COLUMN], ID_COLUMN, OUTPUT / "submission.csv"
)
roundtrip = pd.read_csv(
    destination, converters={ID_COLUMN: str}, keep_default_na=False, float_precision="round_trip"
)
assert roundtrip[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
assert roundtrip.columns.tolist() == sample.columns.tolist()
assert len(roundtrip) == len(sample)
pd.testing.assert_frame_equal(roundtrip.astype(str), submission.astype(str), check_dtype=False)
save_json(
    {"train_ids": tr[ID_COLUMN].tolist(), "validation_ids": va[ID_COLUMN].tolist()},
    OUTPUT / "fold_ids.json",
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>submission, destination = deliver_submission(
    sample, output_values, test[ID_COLUMN], ID_COLUMN, OUTPUT / &quot;submission.csv&quot;
)
roundtrip = pd.read_csv(
    destination, converters={ID_COLUMN: str}, keep_default_na=False, float_precision=&quot;round_trip&quot;
)
assert roundtrip[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
assert roundtrip.columns.tolist() == sample.columns.tolist()
assert len(roundtrip) == len(sample)
pd.testing.assert_frame_equal(roundtrip.astype(str), submission.astype(str), check_dtype=False)
save_json(
    {&quot;train_ids&quot;: tr[ID_COLUMN].tolist(), &quot;validation_ids&quot;: va[ID_COLUMN].tolist()},
    OUTPUT / &quot;fold_ids.json&quot;,
)

</code></pre>
</details>


In [ ]:
save_json(
    {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "max_rows": MAX_ROWS,
        "image_size": IMAGE_SIZE,
        "model_name": MODEL_NAME,
        "imbalance_weights": USE_IMBALANCE_WEIGHTS,
        "pretrained_allowed": PRETRAINED_ALLOWED,
        "initial_weight_path": WEIGHT_PATH,
        "asset_provenance": ASSET_PROVENANCE,
        "trainer": globals().get("training"),
    },
    OUTPUT / "lesson_config.json",
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>save_json(
    {
        &quot;epochs&quot;: EPOCHS,
        &quot;batch_size&quot;: BATCH_SIZE,
        &quot;seed&quot;: SEED,
        &quot;max_rows&quot;: MAX_ROWS,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;model_name&quot;: MODEL_NAME,
        &quot;imbalance_weights&quot;: USE_IMBALANCE_WEIGHTS,
        &quot;pretrained_allowed&quot;: PRETRAINED_ALLOWED,
        &quot;initial_weight_path&quot;: WEIGHT_PATH,
        &quot;asset_provenance&quot;: ASSET_PROVENANCE,
        &quot;trainer&quot;: globals().get(&quot;training&quot;),
    },
    OUTPUT / &quot;lesson_config.json&quot;,
)

</code></pre>
</details>


In [ ]:
record_run(
    OUTPUT,
    task=FAMILY,
    model=MODEL_DESCRIPTION,
    configuration=competition.to_dict(),
    validation_metrics={
        "official_score": float(validation_score),
        "fit_seconds": fit_seconds,
        "inference_seconds": inference_seconds,
    },
    split=split,
    inference_options={
        "threshold": float(selected_threshold),
        "label_order": TARGET_COLUMNS,
        "image_size": IMAGE_SIZE,
        "model_name": MODEL_NAME,
        "asset_provenance": ASSET_PROVENANCE,
    },
    checkpoint=str(checkpoint_path),
    notes=[
        "Educational candidate, not a reproduced Kaggle leaderboard score.",
        "No test labels, test pseudo-labeling or unauthorized external data used.",
        f"Imbalance weights enabled: {USE_IMBALANCE_WEIGHTS}",
    ],
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>record_run(
    OUTPUT,
    task=FAMILY,
    model=MODEL_DESCRIPTION,
    configuration=competition.to_dict(),
    validation_metrics={
        &quot;official_score&quot;: float(validation_score),
        &quot;fit_seconds&quot;: fit_seconds,
        &quot;inference_seconds&quot;: inference_seconds,
    },
    split=split,
    inference_options={
        &quot;threshold&quot;: float(selected_threshold),
        &quot;label_order&quot;: TARGET_COLUMNS,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;model_name&quot;: MODEL_NAME,
        &quot;asset_provenance&quot;: ASSET_PROVENANCE,
    },
    checkpoint=str(checkpoint_path),
    notes=[
        &quot;Educational candidate, not a reproduced Kaggle leaderboard score.&quot;,
        &quot;No test labels, test pseudo-labeling or unauthorized external data used.&quot;,
        f&quot;Imbalance weights enabled: {USE_IMBALANCE_WEIGHTS}&quot;,
    ],
)

</code></pre>
</details>


In [ ]:
pd.DataFrame(
    [
        {
            "run": RUN_ID,
            "family": FAMILY,
            "variant": VARIANT,
            "score": validation_score,
            "fit_seconds": fit_seconds,
            "inference_seconds": inference_seconds,
        }
    ]
).to_csv(OUTPUT / "comparison_row.csv", index=False)
copied = sync_artifacts(OUTPUT, DURABLE_OUTPUT)
print("Checked local CSV:", destination, "Durable files copied:", len(copied))



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>pd.DataFrame(
    [
        {
            &quot;run&quot;: RUN_ID,
            &quot;family&quot;: FAMILY,
            &quot;variant&quot;: VARIANT,
            &quot;score&quot;: validation_score,
            &quot;fit_seconds&quot;: fit_seconds,
            &quot;inference_seconds&quot;: inference_seconds,
        }
    ]
).to_csv(OUTPUT / &quot;comparison_row.csv&quot;, index=False)
copied = sync_artifacts(OUTPUT, DURABLE_OUTPUT)
print(&quot;Checked local CSV:&quot;, destination, &quot;Durable files copied:&quot;, len(copied))

</code></pre>
</details>


In [ ]:
assert destination.is_file()
assert submission[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
print("Delivery complete. Next: restart inference, then attempt the reconstruction task.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert destination.is_file()
assert submission[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
print(&quot;Delivery complete. Next: restart inference, then attempt the reconstruction task.&quot;)

</code></pre>
</details>


### Stop and explain

Which ID-order or encoding mistake would survive a plausible-looking preview? What did the
on-disk round trip prove?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Shuffled sample IDs and literal strings such as 001/NA require explicit alignment and CSV
parsing. Readback checks the delivered file's schema/order/values; official task semantics and
upload acceptance still need separate confirmation.

</details>



## Transfer and closed-book reconstruction

The organizer supplies seven classes and plant_id, and disables pretrained weights. Rebuild the
head and mapping, split by plant, and demonstrate a scratch-only complete submission before
tuning resolution.

1. Close the reference notebook and create an empty personal notebook.
2. Rebuild the data contract, split, features/model, fit, official metric and submission path.
3. Deliver a checked small-pack CSV within 45–60 minutes, then reload it in a fresh session.
4. Record elapsed time, hints, bugs and the first function you could not reconstruct.
5. Rebuild that function the next day with a different schema or seed.

For the 2k assistant, send one function signature, input/output shapes and one failing toy test.
Ask for an explanation or correction of that function only. Keep the rest of the notebook
outside the prompt.

### Controlled comparison

Compare the scratch CNN with EfficientNet at a measured small resolution first. Separately test
augmentation and legally permitted initialization. Keep specimen groups fixed and include full
image inference cost.

A first correct submission is the readiness gate. Advanced complexity is selected only after an
honest metric/runtime comparison; no test fitting or model-weight permission is implied.
